# Ранжирование русскоязычных результатов поиска

Jina Reranker v2 и CatBoostRanker оценивают каждый фрагмент для заданного запроса. Итоговый реранкер учит веса, с которыми складываются эти оценки.

Расчёт проверен на CPU под Linux (Debian 13), Python 3.12.14. Инструкции по запуску и результаты экспериментов — в `README.md`.

## Данные

Читаем `train.csv`, `test.csv` и `sample_submission.csv`. Чистим текст и заполняем пустые заголовки по другим записям той же статьи, когда это возможно.

In [ ]:
from collections import Counter
from functools import lru_cache
from html import unescape
from pathlib import Path
import os
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer


WORD = re.compile(r"[^\W_]+", re.UNICODE)
HTML_TAG = re.compile(r"</?[a-z][a-z0-9]*(?:\s+[^<>]*)?\s*/?>", re.I)
NAV = re.compile(r"^\s*(?:навигация\s*:|главная\s*[|›»>])[^\n]*\n+", re.I)
WIKI = re.compile(r"\s*[-–—|]\s*википедия\s*$", re.I)
SENTENCE = re.compile(r"(?<=[.!?])\s+(?=[А-ЯЁA-Z«„\"(0-9])|\n+|;\s+")
CYR = re.compile(r"[а-я]+$")
VOWEL = re.compile("[аеиоуыэюя]")
STOP = set("а в во вон все всего весь вот где да даже для до его ее ей если есть еще же за и из или им их к как какая какого какое какой каком какую каким кем когда кого кому кто ли либо лишь мне может можно мы на над нам не него нее ней нет ни но ну о об от по под при про с сам себе себя со так такой там те тем то того тоже той только том тот ту ты у уже чем чему что чтоб чтобы чья чье чей чьи это этого этой этом эту являлся является являться был была были было быть бы который которые которого которых которым между через после перед до где куда откуда сколько почему зачем какой когда поскольку".split())
QUESTION_TYPES = (
    ("date", r"\b(?:когда|в каком (?:году|веке|месяце)|какого числа|какая дата|в какие годы)\b"),
    ("amount", r"\b(?:сколько|каков[ао]? (?:вес|рост|численность|длина|площадь|высота|население)|какая (?:длина|высота|площадь|численность))\b"),
    ("place", r"\b(?:где|куда|откуда|в как(?:ом|ой) (?:стране|городе|области|части|регионе|месте))\b"),
    ("person", r"\b(?:кто|кем|кого|кому|чей|чья)\b"),
    ("name", r"\b(?:как (?:звали|называл[аио]?(?:сь|ся)?|называют|называет(?:ся)?)|какое (?:имя|название)|как(?:ой|ая) (?:имя|название))\b"),
    ("reason", r"\b(?:почему|зачем|из-за чего|по какой причине)\b"),
    ("definition", r"\b(?:что (?:такое|означает|значит)|кто такой|кто такая)\b"),
    ("choice", r"\b(?:какой|какая|какое|какие|какого|какую)\b"),
    ("yesno", r"\bли\b|^(?:есть|был[аио]?|является|можно|может)\b"),
)
QUESTION_TYPES = tuple((name, re.compile(pattern)) for name, pattern in QUESTION_TYPES)
ANSWER_PATTERNS = {
    "number": re.compile(r"\d+(?:[.,]\d+)?"),
    "year": re.compile(r"\b(?:[12]\d{3}|\d{1,3}(?=\s*(?:гг?\.|год|до н\.?\s*э)))\b", re.I),
    "month": re.compile(r"\b(?:январ|феврал|март|апрел|ма[йяе]|июн|июл|август|сентябр|октябр|ноябр|декабр)\w*", re.I),
    "century": re.compile(r"\b[IVXLC]+\s+(?:век\w*|вв?\.)|\bвек\w*", re.I),
    "person": re.compile(r"\b[А-ЯЁA-Z][а-яёa-z-]+(?:\s+[А-ЯЁA-Z][а-яёa-z-]+){1,3}\b"),
    "quote": re.compile(r"«[^»]{2,80}»|\"[^\"\n]{2,80}\""),
    "place": re.compile(r"\b(?:город\w*|столиц\w*|стран\w*|област\w*|район\w*|наход\w*|располож\w*)\b", re.I),
    "unit": re.compile(r"%|\b(?:км|кг|млн|млрд|тонн\w*|тысяч\w*|миллион\w*|метр\w*|копий|человек|лет)\b", re.I),
    "negation": re.compile(r"\b(?:не|нет|никогда|невозможно|отсутств\w*|нельзя)\b", re.I),
}
PATTERN_WORDS = {"date": "year", "amount": "number", "place": "place", "person": "person", "name": "quote", "yesno": "negation"}
NOISE = {
    "html": re.compile(r"</?[a-z]|&(?:amp;)*(?:lt|gt);", re.I),
    "navigation": NAV,
    "stress": re.compile(r"\u0301"),
    "invisible": re.compile(r"[\u00ad\u200b-\u200f\ufeff]"),
    "list": re.compile(r"(?:^|\n)\s*(?:[•*#]|\d+[.)])\s"),
}
NUMERIC = ("passage_index", "article_passage_count", "article_char_count")


def find_data(stem):
    custom = os.environ.get("PMMO_DATA_DIR")
    if custom:
        directories = [Path(custom)]
    else:
        directories = [Path.cwd(), Path.cwd() / "data", Path.cwd() / "upload"]
        kaggle = Path("/kaggle/input")
        if kaggle.is_dir():
            roots = sorted({p.parent for p in kaggle.rglob("sample_submission.csv")
                            if (p.parent / "train.csv").is_file() and (p.parent / "test.csv").is_file()})
            if len(roots) == 1:
                directories += roots
    for directory in directories:
        exact = directory / f"{stem}.csv"
        if exact.is_file():
            return exact
        copies = sorted(directory.glob(f"{stem} (*).csv"))
        if len(copies) == 1:
            return copies[0]
        if len(copies) > 1:
            raise ValueError(f"Найдено несколько копий {stem}.csv в {directory}")
    raise FileNotFoundError(f"Положите {stem}.csv в папку запуска, data/ или upload/")


def load_data(stem):
    types = dict.fromkeys(("id", "query_id", "document_id", "article_id", *NUMERIC), "string")
    return pd.read_csv(find_data(stem), dtype=types)


def clean(value):
    value = unicodedata.normalize("NFKC", unescape(unescape(value)))
    value = NOISE["stress"].sub("", value)
    value = NOISE["invisible"].sub("", value)
    value = HTML_TAG.sub(" ", value)
    return NAV.sub("", value.strip()).strip()


@lru_cache(maxsize=300_000)
def stem(word):
    """Отсечение русских окончаний в области после первой гласной."""
    if len(word) <= 3 or CYR.fullmatch(word) is None:
        return word
    first_vowel = VOWEL.search(word)
    if first_vowel is None:
        return word
    head, tail = word[:first_vowel.end()], word[first_vowel.end():]
    tail, gerund = re.subn(r"(?:ивши|ывши|ившись|ывшись|ив|ыв|(?<=[ая])(?:вшись|вши|в))$", "", tail)
    if not gerund:
        tail = re.sub(r"(?:ся|сь)$", "", tail)
        tail, adjective = re.subn(r"(?:ими|ыми|его|ого|ему|ому|ее|ие|ые|ое|ей|ий|ый|ой|ем|им|ым|ом|их|ых|ую|юю|ая|яя|ою|ею)$", "", tail)
        if adjective:
            tail = re.sub(r"(?:ивш|ывш|ующ|(?<=[ая])(?:ем|нн|вш|ющ|щ))$", "", tail)
        else:
            tail, verb = re.subn(r"(?:ила|ыла|ена|ейте|уйте|ите|или|ыли|ей|уй|ил|ыл|им|ым|ен|ило|ыло|ено|ят|ует|уют|ит|ыт|ены|ить|ыть|ишь|ую|ю|(?<=[ая])(?:ла|на|ете|йте|ли|й|л|ем|н|ло|но|ет|ны|ть|ешь|нно))$", "", tail)
            if not verb:
                tail = re.sub(r"(?:иями|ями|ами|ией|иям|ием|иях|ев|ов|ие|ье|еи|ии|ей|ой|ий|ям|ям|ем|ам|ом|ах|ях|ию|ью|ия|ья|а|е|и|й|о|у|ы|ь|ю|я)$", "", tail)
    tail = re.sub(r"и$", "", tail)
    result = head + tail
    transitions = list(re.finditer(r"[аеиоуыэюя][^аеиоуыэюя]", result))
    if len(transitions) >= 2 and result.endswith(("ость", "ост")) and result[transitions[1].end():].endswith(("ость", "ост")):
        result = re.sub(r"ость?$", "", result)
    result = re.sub(r"(?:ейше|ейш)$", "", result)
    result = re.sub(r"(?:н(?=н$)|ь$)", "", result)
    return result if len(result) >= 2 else word


def words(value):
    return tuple(WORD.findall(value.lower().replace("ё", "е")))


@lru_cache(maxsize=120_000)
def word_grams(word):
    padded = f"^{word}$"
    return frozenset(padded[i:i + 3] for i in range(len(word)))

In [ ]:
def normalize_number(series):
    formatted = series.astype("string").str.replace(r"\s+", "", regex=True).str.replace(",", ".", regex=False)
    return pd.to_numeric(formatted, errors="coerce").astype(float)


def article_median(values, keys):
    return values.groupby(keys, sort=False).transform("median")


class PreparedRows:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        data = self.data
        self.q_keys, self.a_keys, self.d_keys = (data[c] for c in ("query_id", "article_id", "document_id"))
        self.qa_keys = [self.q_keys, self.a_keys]
        self.raw_titles = data.title.fillna("")
        self.raw_text = data.text.fillna("")
        self.queries = self.q_keys.map(data.drop_duplicates("query_id").set_index("query_id")["query"].fillna("").map(clean))
        self.text = self.d_keys.map(data.drop_duplicates("document_id").set_index("document_id").text.fillna("").map(clean))
        titles = self.raw_titles.map(clean).str.replace(WIKI, "", regex=True).str.strip()
        observed = pd.DataFrame({"article": self.a_keys, "title": titles})
        votes = observed[observed.title.ne("")].value_counts().reset_index(name="votes")
        votes = votes.sort_values(["votes", "title"], ascending=[False, True])
        canonical = votes.drop_duplicates("article").set_index("article").title
        self.observed_titles = titles
        self.titles = self.a_keys.map(canonical).fillna("").where(titles.eq(""), titles)
        self.query_tokens = self.queries.map(words).tolist()
        self.title_tokens = self.titles.map(words).tolist()
        self.text_tokens = self.d_keys.map(pd.Series(self.text.values, index=self.d_keys).groupby(level=0, sort=False).first().map(words)).tolist()
        self.query_stems, self.title_stems, self.text_stems = (
            [tuple(map(stem, tokens)) for tokens in seq]
            for seq in (self.query_tokens, self.title_tokens, self.text_tokens)
        )
        self.doc_positions = np.flatnonzero(~self.d_keys.duplicated().to_numpy())
        self.article_positions = np.flatnonzero(~self.a_keys.duplicated().to_numpy())
        self.question_type = self.queries.map(self.get_type)

    @staticmethod
    def get_type(query):
        query = query.lower().replace("ё", "е")
        return next((name for name, pattern in QUESTION_TYPES if pattern.search(query)), "other")

In [ ]:
import platform
from time import monotonic

from threadpoolctl import threadpool_limits


train = load_data("train")
test = load_data("test")
sample = load_data("sample_submission")

required = {"id", "query_id", "document_id", "article_id", "query", "title", "text", "ingest_source", *NUMERIC}
assert required | {"relevance"} <= set(train.columns)
assert required <= set(test.columns)
assert set(train.relevance.unique()) <= {0, 1}
assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
assert not set(train.query_id) & set(test.query_id)
assert not set(train.document_id) & set(test.document_id)
assert set(test.id) == set(sample.id)

print(f"{platform.system()}, Python {platform.python_version()}")
print(f"Train — {len(train):,} пар и {train.query_id.nunique():,} запросов")
print(f"Test  — {len(test):,} пар и {test.query_id.nunique():,} запросов")

## Признаки для деревьев

Сравниваем запрос с заголовком и текстом по BM25, общим словам, символьным n-граммам и совпадениям в коротких окнах.

In [ ]:
class CorpusStats:
    def __init__(self, rows):
        self.body = self.build(rows.text_stems, rows.doc_positions)
        self.title = self.build(rows.title_stems, rows.article_positions)

    @staticmethod
    def build(documents, indices):
        df = Counter()
        total_length = 0
        for idx in indices:
            tokens = documents[idx]
            df.update(set(tokens))
            total_length += len(tokens)
        n_docs = len(indices)
        weights = {word: np.log((n_docs + 1) / (freq + 0.5)) for word, freq in df.items()}
        return weights, np.log(n_docs + 1), max(1, total_length / n_docs)


def overlap_features(query, counts, weights, mean_length):
    length = sum(counts.values())
    hits = [counts.get(token, 0) for token in query]
    matched_weight = sum(weight for hit, weight in zip(hits, weights) if hit)
    pivot = 1.2 * (0.25 + 0.75 * length / mean_length)
    bm25 = sum(weight * 2.2 * hit / (pivot + hit) for hit, weight in zip(hits, weights))
    return (bm25, matched_weight / sum(weights), sum(hit > 0 for hit in hits) / len(query), matched_weight)


def fuzzy_weights(query, counts, weights):
    grams = [word_grams(token) for token in query]
    best = [1.0 if token in counts else 0.0 for token in query]
    if not all(best):
        for token in counts:
            if len(token) < 3:
                continue
            candidate_grams = word_grams(token)
            for j, (qtoken, qgrams) in enumerate(zip(query, grams)):
                if best[j] == 1.0 or len(qtoken) < 3 or abs(len(qtoken) - len(token)) > max(3, len(qtoken) // 2):
                    continue
                common = len(qgrams & candidate_grams)
                similarity = 2.0 * common / (len(qgrams) + len(candidate_grams))
                if similarity > best[j]:
                    best[j] = similarity
    return sum(w * score for w, score in zip(weights, best)) / sum(weights)


def phrases(query, field, weights):
    pairs = {(a, b): min(weights[a], weights[b]) for a, b in zip(query, query[1:]) if a != b}
    pairs_found = set(zip(field, field[1:])) & pairs.keys()
    bigram = sum(pairs[p] for p in sorted(pairs_found)) / max(1, sum(pairs.values()))
    core_set = set(query)
    positions = [(i, token) for i, token in enumerate(field) if token in core_set]
    near_pairs = {(a, b) for (i, a), (j, b) in zip(positions, positions[1:]) if j - i <= 4} & pairs.keys()
    near = sum(pairs[p] for p in sorted(near_pairs)) / max(1, sum(pairs.values()))
    return bigram, near, positions


def best_window(positions, weights, size):
    left, total, best = 0, 0.0, 0.0
    seen = Counter()
    for index, token in positions:
        seen[token] += 1
        if seen[token] == 1:
            total += weights[token]
        while positions[left][0] <= index - size:
            old = positions[left][1]
            seen[old] -= 1
            if seen[old] == 0:
                total -= weights[old]
            left += 1
        best = max(best, total)
    return best / sum(weights.values())


def named_tokens(query):
    token_strings = WORD.findall(query)
    quoted = [words(match.group()) for match in ANSWER_PATTERNS["quote"].finditer(query)]
    quoted_tokens = {word for group in quoted for word in group}
    return tuple(dict.fromkeys(stem(token.lower().replace("ё", "е")) for i, token in enumerate(token_strings)
                              if (i > 0 and token[:1].isupper()) or token.lower().replace("ё", "е") in quoted_tokens))

In [ ]:
def lexical_features(rows, corpus, advanced):
    names = (
        "bm25_body", "coverage_body", "share_body", "matched_body",
        "bm25_title", "coverage_title", "share_title", "matched_title",
        "coverage_both", "body_both_share", "core_coverage_body", "core_coverage_title", "core_coverage_both",
        "title_in_query", "rarest_body", "rarest_title", "raw_coverage_body", "raw_coverage_title",
        "query_len", "query_idf", "body_len", "title_len",
    )
    advanced_names = (
        "fuzzy_body", "fuzzy_title", "bigram_body", "near_body", "bigram_title", "near_title", "window_12", "window_36",
        "first_hit", "head_coverage", "name_body", "name_title", "name_both", "title_in_head", "title_in_body",
        "sentence_max", "sentence_mean", "sentence_answer", "sentence_near", "sentence_density", "sentence_count",
        "quote_body", "quote_title", "number_body", "number_title", "answer_closest", "answer_at_start",
    )
    records = []
    idf_b, unseen_b, mean_b = corpus.body
    idf_t, unseen_t, mean_t = corpus.title
    for i, (qraw, qstem, title, body) in enumerate(zip(rows.query_tokens, rows.query_stems, rows.title_stems, rows.text_stems)):
        query = tuple(dict.fromkeys(qstem)) or ("",)
        body_counts, title_counts = Counter(body), Counter(title)
        weights = {token: idf_b.get(token, unseen_b) for token in query}
        all_weights = list(weights.values())
        title_weights = [idf_t.get(token, unseen_t) for token in query]
        core = tuple(dict.fromkeys(s for w, s in zip(qraw, qstem) if w not in STOP and len(w) > 1)) or query
        core_weight = {token: weights[token] for token in core}
        core_total = sum(core_weight.values())
        union = body_counts.keys() | title_counts.keys()
        total = sum(all_weights)
        rarest = max(core, key=weights.get)
        raw_q = set(qraw)
        raw_b, raw_t = set(rows.text_tokens[i]), set(rows.title_tokens[i])
        record = [*overlap_features(query, body_counts, all_weights, mean_b),
                  *overlap_features(query, title_counts, title_weights, mean_t),
                  sum(weights[t] for t in query if t in union) / total,
                  sum(weights[t] for t in query if t in body_counts and t in title_counts) / total,
                  sum(weights[t] for t in core if t in body_counts) / core_total,
                  sum(weights[t] for t in core if t in title_counts) / core_total,
                  sum(weights[t] for t in core if t in union) / core_total,
                  len(set(title) & set(query)) / max(1, len(set(title))),
                  body_counts[rarest], title_counts[rarest], len(raw_q & raw_b) / max(1, len(raw_q)),
                  len(raw_q & raw_t) / max(1, len(raw_q)), len(qraw), total / len(query), len(body), len(title)]
        if advanced:
            ordered_core = tuple(s for w, s in zip(qraw, qstem) if w not in STOP and len(w) > 1) or query
            big_b, near_b, positions = phrases(ordered_core, body, core_weight)
            big_t, near_t, _ = phrases(ordered_core, title, core_weight)
            first = positions[0][0] / max(1, len(body)) if positions else 1.0
            head = set(body[:36])
            title_content = set(title) - STOP
            entities = named_tokens(rows.queries.iat[i])
            entity_total = max(1, len(entities))
            text = rows.text.iat[i]
            sentences = [s for s in SENTENCE.split(text) if s.strip()]
            answer = ANSWER_PATTERNS.get(PATTERN_WORDS.get(rows.question_type.iat[i], ""))
            coverage, answer_coverage, answer_distance, density = [], [], [], []
            for sentence in sentences:
                sentence_tokens = tuple(map(stem, words(sentence)))
                sentence_set = set(sentence_tokens)
                score = sum(core_weight[t] for t in core if t in sentence_set) / core_total
                coverage.append(score)
                density.append(score / np.sqrt(max(5, len(sentence_tokens))))
                if answer is not None and answer.search(sentence):
                    answer_coverage.append(score)
                    for match in answer.finditer(sentence):
                        answer_distance.extend(abs(match.start() - word.start()) for word in WORD.finditer(sentence)
                                               if stem(word.group().lower().replace("ё", "е")) in core_weight)
            qnumbers = {t for t in qraw if t.isdigit()}
            query_quotes = [" ".join(words(match.group())) for match in ANSWER_PATTERNS["quote"].finditer(rows.queries.iat[i])]
            text_join, title_join = " ".join(rows.text_tokens[i]), " ".join(rows.title_tokens[i])
            record.extend((fuzzy_weights(core, body_counts, list(core_weight.values())),
                           fuzzy_weights(core, title_counts, list(core_weight.values())),
                           big_b, near_b, big_t, near_t, best_window(positions, core_weight, 12),
                           best_window(positions, core_weight, 36), first,
                           sum(core_weight[t] for t in core if t in head) / core_total,
                           sum(t in body_counts for t in entities) / entity_total,
                           sum(t in title_counts for t in entities) / entity_total,
                           sum(t in union for t in entities) / entity_total,
                           len(title_content & head) / max(1, len(title_content)),
                           len(title_content & body_counts.keys()) / max(1, len(title_content)),
                           max(coverage, default=0.0), sum(coverage) / max(1, len(coverage)),
                           max(answer_coverage, default=0.0), sum(answer_coverage) / max(1, len(coverage)),
                           max(density, default=0.0), len(sentences),
                           sum(q in text_join for q in query_quotes if q), sum(q in title_join for q in query_quotes if q),
                           len(qnumbers & raw_b), len(qnumbers & raw_t),
                           np.log1p(min(answer_distance)) if answer_distance else np.nan,
                           float(bool(answer and sentences and answer.search(sentences[0])))))
        records.append(record)
    return pd.DataFrame(records, columns=names + (advanced_names if advanced else ()), dtype=np.float32)

In [ ]:
def row_cosine(left, right):
    return np.asarray(left.multiply(right).sum(axis=1)).ravel()


class CharacterSimilarity:
    def __init__(self, rows):
        self.vectorizer = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2**17,
                                            alternate_sign=False, norm=None, lowercase=False, dtype=np.float32)
        documents = (" ".join(rows.text_tokens[i]) for i in rows.doc_positions)
        self.tfidf = TfidfTransformer(sublinear_tf=True).fit(self.vectorizer.transform(documents))

    def transform(self, documents):
        return self.tfidf.transform(self.vectorizer.transform(documents))

    def features(self, rows):
        parts = []
        for start in range(0, len(rows.data), 1500):
            stop = start + 1500
            query = self.transform(map(" ".join, rows.query_tokens[start:stop]))
            core = self.transform(" ".join(t for t in tokens if t not in STOP) for tokens in rows.query_tokens[start:stop])
            title = self.transform(map(" ".join, rows.title_tokens[start:stop]))
            body = self.transform(map(" ".join, rows.text_tokens[start:stop]))
            head = self.transform(" ".join(t[:36]) for t in rows.text_tokens[start:stop])
            parts.append(np.column_stack([row_cosine(query, body), row_cosine(query, title), row_cosine(core, body),
                                          row_cosine(core, title), row_cosine(core, head), row_cosine(title, body),
                                          row_cosine(title, head)]))
        return pd.DataFrame(np.vstack(parts), columns=("char_body", "char_title", "char_core_body", "char_core_title",
                                                       "char_head", "char_title_body", "char_title_head"))

### Метаданные и соседние фрагменты

Отмечаем пропуски и несогласованные позиции. Сравниваем кандидатов внутри запроса и фрагменты внутри статьи.

In [ ]:
def metadata_features(rows):
    data = rows.data
    idx, count, chars = [normalize_number(data[column]) for column in NUMERIC]
    index = idx.where(idx.ge(0))
    passage_count = count.where(count.ge(1))
    article_chars = chars.where(chars.ge(1))
    count_consensus = article_median(passage_count, rows.a_keys)
    chars_consensus = article_median(article_chars, rows.a_keys)
    count_used = passage_count.fillna(count_consensus)
    chars_used = article_chars.fillna(chars_consensus)
    size = rows.q_keys.groupby(rows.q_keys, sort=False).transform("size")
    article_size = rows.q_keys.groupby(rows.qa_keys, sort=False).transform("size")
    n_articles = rows.a_keys.groupby(rows.q_keys, sort=False).transform("nunique")
    index_by_doc = article_median(index, rows.d_keys)
    index_used = index.fillna(index_by_doc)
    out = {
        "index": np.log1p(index_used), "article_count": np.log1p(count_used), "article_chars": np.log1p(chars_used),
        "index_share": index_used.div(count_used.where(index_used.lt(count_used))),
        "mean_fragment_chars": np.log1p(chars_used.div(count_used)),
        "index_missing": index.isna(), "count_missing": passage_count.isna(), "chars_missing": article_chars.isna(),
        "index_inconsistent": index_used.ge(count_used), "count_consensus": np.log1p(count_consensus),
        "chars_consensus": np.log1p(chars_consensus), "title_missing": rows.observed_titles.eq(""),
        "text_chars": np.log1p(rows.text.str.len()), "title_chars": np.log1p(rows.titles.str.len()),
        "query_size": size, "query_articles": n_articles, "same_article": article_size, "article_share": article_size / size,
        "index_from_first": np.log1p(index_used - index_used.groupby(rows.qa_keys, sort=False).transform("min")),
        "index_to_last": np.log1p(index_used.groupby(rows.qa_keys, sort=False).transform("max") - index_used),
        "seen_doc_count": np.log1p(rows.d_keys.groupby(rows.d_keys, sort=False).transform("size")),
        "seen_article_count": np.log1p(rows.q_keys.groupby(rows.a_keys, sort=False).transform("nunique")),
        "word_upper": rows.text.str.count(r"[A-ZА-ЯЁ]") / rows.text.str.len().clip(lower=1),
        "digits_share": rows.text.str.count(r"\d") / rows.text.str.len().clip(lower=1),
        "parentheses": rows.text.str.count(r"\("), "newlines": rows.text.str.count(r"\n"),
        "dash_start": rows.text.str.slice(0, 140).str.contains(r"\s[—–]\s", regex=True),
    }
    for name in ("S0", "S1", "S2", "S3"):
        out[f"source_{name}"] = data.ingest_source.eq(name)
    for name, _ in QUESTION_TYPES:
        out[f"qtype_{name}"] = rows.question_type.eq(name)
    for name, pattern in ANSWER_PATTERNS.items():
        out[f"answer_{name}"] = np.log1p(rows.text.str.count(pattern))
    for name, pattern in NOISE.items():
        out[f"format_{name}"] = rows.raw_text.str.contains(pattern)
    return pd.DataFrame(out, dtype=np.float32)


RELATIVE_COLUMNS = ("bm25_body", "bm25_title", "core_coverage_body", "core_coverage_title", "core_coverage_both",
                    "char_core_body", "char_core_title", "sentence_max", "window_12", "fuzzy_body", "fuzzy_title",
                    "title_in_query", "article_count", "same_article", "text_chars", "seen_doc_count")


def relative_features(rows, features):
    result = {}
    for name in RELATIVE_COLUMNS:
        if name not in features:
            continue
        value = features[name]
        query = value.groupby(rows.q_keys, sort=False)
        article = value.groupby(rows.qa_keys, sort=False)
        result[f"q_rank_{name}"] = query.rank(method="average", pct=True)
        result[f"q_max_gap_{name}"] = query.transform("max") - value
        result[f"q_center_{name}"] = value - query.transform("mean")
        result[f"a_max_gap_{name}"] = article.transform("max") - value
        result[f"a_max_{name}"] = article.transform("max")
    return pd.DataFrame(result, dtype=np.float32)



def build_features(rows, corpus, characters):
    features = pd.concat([lexical_features(rows, corpus, advanced=True),
                          characters.features(rows), metadata_features(rows)], axis=1)
    return pd.concat([features, relative_features(rows, features)], axis=1)

In [ ]:
from scipy.sparse import coo_matrix
from sklearn.preprocessing import normalize
from sklearn.utils.extmath import randomized_svd

def document_context(rows, corpus):
    idf, unseen, _ = corpus.body
    out = np.zeros((len(rows.data), 21), dtype=np.float32)
    text_sets = [set(t) - STOP for t in rows.text_stems]
    title_sets = [set(t) - STOP for t in rows.title_stems]
    text_pairs = [set(zip(t, t[1:])) for t in rows.text_stems]
    positions = normalize_number(rows.data.passage_index).to_numpy()
    articles = rows.a_keys.to_numpy()
    for group in rows.data.groupby("query_id", sort=False).indices.values():
        q0 = group[0]
        q = tuple(dict.fromkeys(t for t, raw in zip(rows.query_stems[q0], rows.query_tokens[q0]) if raw not in STOP))
        group_text = [text_sets[i] for i in group]
        group_title = [title_sets[i] for i in group]
        text_frequency, title_frequency = Counter(), Counter()
        for text, title in zip(group_text, group_title):
            text_frequency.update(token for token in q if token in text)
            title_frequency.update(token for token in q if token in title)
        rarity = np.array([np.log1p((len(group) + 1) / (text_frequency[t] + 1)) for t in q])
        combined = rarity * [idf.get(t, unseen) for t in q]
        common = np.array([text_frequency[t] / len(group) for t in q])
        norm = max(1.0, combined.sum())
        terms = np.array([[t in text for t in q] for text in group_text], dtype=float)
        title_terms = np.array([[t in title for t in q] for title in group_title], dtype=float)
        out[group, 0] = (terms @ combined) / norm
        out[group, 1] = (title_terms @ combined) / norm
        out[group, 2] = (np.maximum(terms, title_terms) @ combined) / norm
        out[group, 3] = (terms @ common) / max(1.0, common.sum())
        out[group, 4] = sum(f == 1 for f in text_frequency.values()) / max(1, len(q))
        out[group, 5] = sum(f == len(group) for f in text_frequency.values()) / max(1, len(q))
        for i, row in enumerate(group):
            text, title = group_text[i], group_title[i]
            same, different = [], []
            similarities, shared_bigrams, title_cross = [], [], []
            for j, other in enumerate(group):
                if i == j:
                    continue
                intersection = text & group_text[j]
                score = len(intersection) / max(1, len(text | group_text[j]))
                similarities.append(score)
                if articles[row] == articles[other]:
                    same.append(score)
                else:
                    different.append(score)
                shared_bigrams.append(len(text_pairs[row] & text_pairs[other]) /
                                      max(1, min(len(text_pairs[row]), len(text_pairs[other]))))
                title_cross.append(len(title & group_text[j]) / max(1, len(title)))
            neighbors = group[(articles[group] == articles[row]) & (group != row)]
            distances = np.abs(positions[neighbors] - positions[row])
            valid_distances = distances[np.isfinite(distances)]
            out[row, 6:] = (
                max(similarities, default=0), sum(similarities) / max(1, len(similarities)),
                max(same, default=0), max(different, default=0), sum(s > 0.32 for s in similarities),
                max(shared_bigrams, default=0), sum(s > 0.2 for s in shared_bigrams),
                max(title_cross, default=0), sum(title_cross) / max(1, len(title_cross)),
                np.log1p(valid_distances.min()) if valid_distances.size else np.nan,
                len(text) / max(1, len(rows.text_tokens[row])), len(text - set(q)) / max(1, len(text)),
                sum(title_frequency[t] for t in q if t in title) / max(1, len(group) * len(q)),
                sum(text_frequency[t] <= 2 for t in q if t in text),
                sum(t not in text and t in title for t in q),
            )
    names = ("q_rarity_body", "q_rarity_title", "q_rarity_union", "q_common_body", "q_unique_terms", "q_universal_terms",
             "neighbor_max", "neighbor_mean", "neighbor_same_article", "neighbor_other_article", "neighbor_similar_count",
             "neighbor_bigram_max", "neighbor_bigram_count", "title_in_other_max", "title_in_other_mean", "passage_nearest",
             "lexical_diversity", "non_query_share", "title_term_frequency", "q_rare_hits", "only_in_title")
    return pd.DataFrame(out, columns=names)

### Сходство слов по контексту

Из совместных появлений слов в обучающих фрагментах строим матрицу PPMI. Усечённое SVD даёт векторы слов, по которым сравниваем запрос с заголовком и текстом.

In [ ]:
class DistributionalSimilarity:
    """Сходство слов по их контекстам во фрагментах обучающего корпуса."""

    def __init__(self, rows, corpus):
        self.idf, self.unseen, _ = corpus.body
        freq = Counter()
        for index in rows.doc_positions:
            freq.update(rows.text_stems[index])
        vocab = [word for word, count in sorted(freq.items(), key=lambda item: (-item[1], item[0]))
                 if count >= 4 and word not in STOP and len(word) >= 2 and not word.isdigit()][:24_000]
        self.vocab = {word: i for i, word in enumerate(vocab)}
        column, row_ids = [], []
        segment = 0
        for idx in rows.doc_positions:
            tokens = rows.text_stems[idx]
            for start in range(0, max(1, len(tokens) - 11), 12):
                window = set(tokens[start:start + 24]) & self.vocab.keys()
                ids = sorted(self.vocab[token] for token in window)
                if len(ids) >= 2:
                    column.extend(ids)
                    row_ids.extend([segment] * len(ids))
                    segment += 1
        matrix = coo_matrix((np.ones(len(column), dtype=np.float32), (row_ids, column)), shape=(segment, len(vocab))).tocsr()
        cooc = matrix.T @ matrix[:, :min(14_000, len(vocab))]
        cooc.setdiag(0)
        cooc.eliminate_zeros()
        row_sum = np.asarray(cooc.sum(axis=1)).ravel()
        context = np.asarray(cooc.sum(axis=0)).ravel() ** 0.75
        cooc = cooc.tocoo()
        ppmi = np.maximum(0, np.log(cooc.data * context.sum() / (row_sum[cooc.row] * context[cooc.col])))
        ppmi *= cooc.data / (cooc.data + 3.0)
        shifted = coo_matrix((ppmi, (cooc.row, cooc.col)), shape=cooc.shape).tocsr()
        shifted.eliminate_zeros()
        left, scale, _ = randomized_svd(shifted, n_components=64, n_iter=4, random_state=2026)
        self.vectors = normalize(left * np.sqrt(scale))

    def field(self, tokens):
        field = [token for token in dict.fromkeys(tokens) if token in self.vocab]
        vecs = self.vectors[[self.vocab[token] for token in field]]
        weights = np.array([self.idf.get(token, self.unseen) for token in field], dtype=np.float32)
        return field, vecs, weights

    def similarity(self, q, d):
        qwords, qvec, weights = q
        dwords, dvec, dweights = d
        if not qwords or not dwords:
            return (0.0, 0.0, 0.0)
        qcenter, dcenter = weights @ qvec, dweights @ dvec
        cosine = qcenter @ dcenter / max(1e-8, np.linalg.norm(qcenter) * np.linalg.norm(dcenter))
        best = np.max(qvec @ dvec.T, axis=1)
        missing = ~np.isin(qwords, dwords)
        return (cosine, weights @ best / weights.sum(), weights[missing] @ best[missing] / max(1.0, weights[missing].sum()))

    def features(self, rows):
        records = []
        for i, (q, title, body) in enumerate(zip(rows.query_stems, rows.title_stems, rows.text_stems)):
            raw = rows.query_tokens[i]
            query = self.field(token for token, orig in zip(q, raw) if orig not in STOP)
            records.append((*self.similarity(query, self.field(title)), *self.similarity(query, self.field(body)),
                            *self.similarity(query, self.field(body[:40]))))
        columns = [f"context_{where}_{measure}" for where in ("title", "body", "head") for measure in ("cos", "best", "unmatched")]
        return pd.DataFrame(records, columns=columns, dtype=np.float32)

### Признаки ответа

Для частых типов вопросов ищем рядом с совпавшими словами даты, числа и имена.

In [ ]:
CUES = (
    r"\b(?:роди[лтв]|рождени|урожене)\w*",
    r"\b(?:умер|скончал|погиб|смерт|кончин|уби[лт])\w*",
    r"\b(?:выпус[кт]|издан|издава|опубликов|релиз|премь[её]р|выш(?:ел|л))\w*",
    r"\b(?:основ[аы]|учреди|образова|созда|возник|откры[лт]|появи)\w*",
    r"\b(?:переимен|названи|имено[вв]|назван|называ|известн.*?как)\w*",
    r"\b(?:побед|выигра|чемпион|одерж|лауреат|наград|заня[лт])\w*",
    r"\b(?:закры|прекрат|ликвид|упраздн|роспуск|распад)\w*",
    r"\b(?:вступ|вош[еёл]|вошла|присоед|принят|членство)\w*",
    r"\b(?:автор|написа|произведен|сочини|создател)\w*",
    r"\b(?:столиц|располож|наход|местополож|гранич|раскин)\w*",
    r"\b(?:первый|первое|первая|первые|впервые|самый ранн|дебют|изначальн)\w*",
    r"\b(?:последний|последнее|последняя|заключительн|финальн)\w*",
    r"\b(?:причин|благодар[яи]|вследствие|обуслов|из-за|потому что|поэтому|поскольку)\w*",
    r"\b(?:супруг|жен[иаиу]|замуж|муж[а-я]*\b|брак|пожени)\w*",
    r"\b(?:руковод|возглав|директор|президент|глава|начальник|председат|командова)\w*",
)
CUES = tuple(re.compile(p, re.I) for p in CUES)
QUESTION_ANCHORS = {"date": "year", "amount": "number", "place": "place", "person": "person", "name": "quote"}
YEAR = re.compile(r"\b(?:[12][0-9]{3})\b")


def question_features(rows, corpus):
    names = (
        "relation_body", "relation_title", "relation_sentence", "relation_answer", "entity_sentence", "entity_answer",
        "cue_overlap", "cue_title", "cue_at_start", "cue_with_answer", "cue_query_count", "type_anchor_count",
        "year_first", "year_min", "year_max", "year_distance", "title_contained", "title_body_contained",
        "question_has_entity", "question_has_year", "max_short_coverage", "max_weighted_relation",
    )
    idf, unseen, _ = corpus.body
    records = []
    for i, (query, title, text) in enumerate(zip(rows.queries, rows.titles, rows.text)):
        qwords = rows.query_tokens[i]
        entities = set(named_tokens(query))
        relation = tuple(dict.fromkeys(s for w, s in zip(qwords, rows.query_stems[i]) if w not in STOP and s not in entities))
        weights = {token: idf.get(token, unseen) for token in relation}
        total = max(1.0, sum(weights.values()))
        anchors = ANSWER_PATTERNS.get(QUESTION_ANCHORS.get(rows.question_type.iat[i]))
        cues = [pattern for pattern in CUES if pattern.search(query)]
        relation_sent, answer_sent, entity_sent, entity_answer = [], [], [], []
        best_density, weighted_relation, cue_with_answer = 0.0, 0.0, False
        segments = [s.strip() for s in SENTENCE.split(text) if s.strip()]
        for segment in segments:
            tokens = set(map(stem, words(segment)))
            share = sum(weight for word, weight in weights.items() if word in tokens) / total
            entity_share = len(entities & tokens) / max(1, len(entities))
            relation_sent.append(share)
            entity_sent.append(entity_share)
            best_density = max(best_density, (share + entity_share) / np.sqrt(max(8, len(tokens))))
            weighted_relation = max(weighted_relation, share * (0.25 + 0.75 * entity_share) if entities else share)
            if anchors is not None and anchors.search(segment):
                answer_sent.append(share)
                entity_answer.append(entity_share)
                cue_with_answer |= any(pattern.search(segment) for pattern in cues)
        years = [int(m.group()) for m in YEAR.finditer(text)]
        q_years = [int(m.group()) for m in YEAR.finditer(query)]
        year_distance = min((abs(a - b) for a in years for b in q_years), default=np.nan)
        short_title = re.sub(r"\s*\([^()]+\)\s*$", "", title)
        short_tokens = tuple(map(stem, words(short_title)))
        title_phrase = " ".join(short_tokens)
        query_phrase = f" {' '.join(rows.query_stems[i])} "
        body_phrase = f" {' '.join(rows.text_stems[i][:100])} "
        bset, tset = set(rows.text_stems[i]), set(rows.title_stems[i])
        records.append((
            sum(w for t, w in weights.items() if t in bset) / total,
            sum(w for t, w in weights.items() if t in tset) / total,
            max(relation_sent, default=0), max(answer_sent, default=0), max(entity_sent, default=0), max(entity_answer, default=0),
            sum(bool(p.search(text)) for p in cues), sum(bool(p.search(title)) for p in cues),
            sum(bool(segments and p.search(segments[0])) for p in cues), cue_with_answer, len(cues),
            np.log1p(len(anchors.findall(text))) if anchors else 0,
            years[0] if years else np.nan, min(years, default=np.nan), max(years, default=np.nan), np.log1p(year_distance),
            bool(title_phrase and f" {title_phrase} " in query_phrase), bool(title_phrase and f" {title_phrase} " in body_phrase),
            bool(entities), bool(q_years), best_density, weighted_relation,
        ))
    out = pd.DataFrame(records, columns=names, dtype=np.float32)
    for col in ("year_first", "year_min", "year_max", "relation_sentence", "max_weighted_relation", "cue_overlap"):
        out[f"rank_{col}"] = out[col].groupby(rows.q_keys, sort=False).rank(pct=True, method="average")
    return out

### Веса пар для объединения скоров

Вес пары зависит от того, насколько изменится NDCG@10, если поменять местами двух кандидатов одного запроса.


In [ ]:
DISCOUNTS = np.r_[1 / np.log2(np.arange(2, 12)), 0.0]


class QueryGroups:
    """Матрица позиций кандидатов. Для равных скоров действует порядок id."""

    def __init__(self, frame):
        ordered = frame.reset_index(drop=True).sort_values(["query_id", "id"])
        sizes = ordered.groupby("query_id", sort=False).size().to_numpy()
        self.valid = np.arange(sizes.max())[None, :] < sizes[:, None]
        self.indices = np.full(self.valid.shape, len(frame), dtype=np.int32)
        self.indices[self.valid] = ordered.index.to_numpy()
        self.size = len(frame)

    def as_matrix(self, values, fill=0):
        return np.r_[values, fill][self.indices]

    def ranking(self, scores):
        return np.argsort(-self.as_matrix(scores, -np.inf), axis=1, kind="stable")


class RelevancePairs:
    """Пары кандидатов с разной релевантностью внутри запроса."""

    def __init__(self, frame):
        self.groups = QueryGroups(frame)
        label = self.groups.as_matrix(frame.relevance.to_numpy())
        group, positive, negative = np.nonzero((label[:, :, None] > label[:, None, :]) & self.groups.valid[:, None, :])
        self.group, self.positive, self.negative = group, positive, negative
        self.pos_index = self.groups.indices[group, positive]
        self.neg_index = self.groups.indices[group, negative]
        ideal = np.r_[0.0, DISCOUNTS[:10].cumsum()][label.sum(axis=1).clip(0, 10)]
        self.inv_ideal = 1.0 / ideal[group]


In [ ]:
def normalize_scores(frame, scores):
    groups = pd.Series(scores).groupby(frame.query_id, sort=False)
    scale = groups.transform("std", ddof=0).clip(lower=0.15)
    return ((scores - groups.transform("mean")) / scale).to_numpy()

In [ ]:
def full_features(rows, corpus, characters, semantic):
    return pd.concat([
        build_features(rows, corpus, characters),
        document_context(rows, corpus),
        semantic.features(rows),
        question_features(rows, corpus),
    ], axis=1)

### Контекст и размеченные примеры

Учитываем другие фрагменты статьи и похожие примеры из обучающего набора. Признаки на основе меток считаем по другим компонентам или фолдам.

In [ ]:
from collections import Counter, defaultdict
import re

import numpy as np
import pandas as pd



INTENT_RULES = (
    (r"\b(?:роди[лт]|рождени|родом)\w*", r"\b(?:роди[лт]|рождени|урожен|появил\w* на свет)\w*"),
    (r"\b(?:умер|погиб|скончал|смерт|уби[лт]|казн)\w*", r"\b(?:умер|погиб|скончал|смерт|уби[лт]|казн|повеш|расстрел|кончин)\w*"),
    (r"\b(?:выш[елла]|выпус[кт]|премь[её]р|релиз)\w*", r"\b(?:выш[елла]|выпус[кт]|изда[лтн]|премь[её]р|релиз|опублико|появил)\w*"),
    (r"\b(?:созда|основа|откры|возник|постро|образова|учреди)\w*", r"\b(?:основа|созда|возник|постро|откры|образова|учреди|появи|залож|учрежд)\w*"),
    (r"\b(?:переименова|назва|называ)\w*", r"\b(?:назва|называ|переименова|переименов|именовал|известн\w* как|получ\w+ имя)\w*"),
    (r"\b(?:закры|распал|распад|прекрат|ликвид)\w*", r"\b(?:закры|распал|распад|прекрат|ликвид|упраздн|роспуск)\w*"),
    (r"\b(?:вступ|присоед|вош[еёл]|член)\w*", r"\b(?:вступ|присоед|вош[еёл]|приня|стал\w? член|членство)\w*"),
    (r"\b(?:впада|впаден|устье)\w*", r"\b(?:впада|впаден|устье|влива|стека|излива|приток)\w*"),
    (r"\b(?:исток|бер[её]т начал|начинает\w* рек)\w*", r"\b(?:исток|бер[её]т начал|вытека|берущ|зарожда)\w*"),
    (r"\b(?:столиц|административн\w* центр)\w*", r"\b(?:столиц|административн\w* центр|главн\w* город|правительств\w* наход)\w*"),
    (r"\b(?:валют|денежн\w+ единиц)\w*", r"\b(?:валют|денежн\w+ единиц|деньги|банкнот|монет|риал|рубл|доллар|франк|фунт|песо)\w*"),
    (r"\b(?:официальн\w+ язык|говорят на|язык\w+ говорят)\w*", r"\b(?:язык|говорят|государственн\w+|официальн\w+)\w*"),
    (r"\b(?:сколько лет жив|сколько .*?жив[еёу]|продолжительност\w+ жизни|срок жизни|в среднем жив)\w*", r"\b(?:жив[еёуи]|дожива|дожить|продолжительност\w+ жизни|возраст)\w*"),
    (r"\b(?:сколько .*?(?:дли[лт]|продолжа)|продолжительност|сколько времени|какой срок)\w*", r"\b(?:дли[лт]|продолжа|продолжительност|длиной|дней|суток|месяц|час|недел|лет)\w*"),
    (r"\b(?:населени|сколько .*?человек|жител|численност)\w*", r"\b(?:населени|жител|численност|человек|прожива|перепис)\w*"),
    (r"\b(?:высот|высок|глубин|глубок|длин|ширин|расстоян|протяж[её]н|площад)\w*", r"\b(?:высот|глубин|длин|ширин|расстоян|протяж[её]н|площад|км|метр|гектар)\w*"),
    (r"\b(?:сколько .*?(?:сезон|эпизод|серий|частей|фильм|том|книг))\w*", r"\b(?:сезон|эпизод|серий|сери[яию]|част|фильм|том|книг|трилоги|всего)\w*"),
    (r"\b(?:сколько .*?(?:звук|букв)|алфавит|фонем)\w*", r"\b(?:звук|букв|фонем|алфавит|гласн|согласн)\w*"),
    (r"\b(?:сколько .*?вес|вес|масс|килограмм)\w*", r"\b(?:вес|масс|веси|кг|тонн|грамм)\w*"),
    (r"\b(?:разработ|изобр[её]л|придумал|изобрет)\w*", r"\b(?:разработ|изобр[её]л|придумал|изобрет|созда|спроектир|студи|компани)\w*"),
    (r"\b(?:написа|автор|сочини)\w*", r"\b(?:написа|автор|сочини|пьеса|роман|текст|произведение)\w*"),
    (r"\b(?:акт[её]р|сыграл|играл\w* роль|снял\w* в)\w*", r"\b(?:акт[её]р|сыграл|исполни|роль|снял|в ролях|каст)\w*"),
    (r"\b(?:режисс[её]р|кто снял|поставил фильм)\w*", r"\b(?:режисс[её]р|постановщик|снял|поставил|экранизир)\w*"),
    (r"\b(?:побед|выигра|перв\w+ мест|чемпион)\w*", r"\b(?:побед|выигра|перв\w+ мест|чемпион|одержал|второе место)\w*"),
    (r"\b(?:президент|правил|власти|правлен|царствова|корол[её]м|император)\w*", r"\b(?:президент|прави|власт|правлен|царствова|корол|император|возглав|должност)\w*"),
    (r"\b(?:жен[аиоу]|супруг|муж|замуж|женат)\w*", r"\b(?:жен[аиоу]|супруг|муж|брак|замуж|женил|пожен)\w*"),
    (r"\b(?:как перевод|означа|значен|расшифров)\w*", r"\b(?:означа|перевод|буквально|значен|расшифров|дословно|от слов)\w*"),
    (r"\b(?:перв\w+|дебют|впервые)\b", r"\b(?:перв\w+|дебют|впервые|самый ранн\w+|поначалу|изначальн\w+)\b"),
    (r"\b(?:почему|по какой причине|зачем)\b", r"\b(?:потому что|поскольку|поэтому|из-за|благодар[яи]|причин\w*|связано с|обуслов\w*)\b"),
)
INTENT_RULES = tuple((re.compile(q, re.I), re.compile(d, re.I)) for q, d in INTENT_RULES)
VALUE_PATTERN = re.compile(r"\b(?:\d{1,3}(?:[ \u00a0]\d{3})+|\d+)(?:[.,]\d+)?\b")
LIFE_DATES = re.compile(r"\([^()]{0,110}\b([12]\d{3})\b[^()]{0,60}[—–−]\s*[^()]{0,55}\b([12]\d{3})\b[^()]{0,60}\)")
ANSWER_UNITS = re.compile(r"\b(?:лет|год\w*|месяц\w*|дней|дня|час\w*|секунд\w*|век\w*|эпизод\w*|сезон\w*|серий|серии|част\w*|км|м|см|кг|тонн\w*|гектар\w*|метр\w*|человек|жител\w*|тыс\.?|миллион\w*|млн\.?|млрд\.?)\b|%", re.I)


class SoftWordMatches:
    """Матрица совпадений между словами запроса и словами кандидата."""

    def __init__(self, tokens):
        self.tokens = tokens
        self.size = len(tokens)
        self.exact = defaultdict(list)
        self.inverted = defaultdict(list)
        self.gram_lengths = np.array([len(word_grams(token)) for token in tokens], dtype=np.float32)
        for i, token in enumerate(tokens):
            self.exact[stem(token)].append(i)
            if len(token) >= 3 and not token.isdigit():
                for gram in word_grams(token):
                    self.inverted[gram].append(i)
        self.cache = {}

    def match(self, token):
        if token not in self.cache:
            strength = np.zeros(self.size, dtype=np.float32)
            if len(token) >= 3 and not token.isdigit():
                grams = word_grams(token)
                for gram in grams:
                    for i in self.inverted.get(gram, ()):
                        strength[i] += 1
                strength *= 2.0 / (len(grams) + self.gram_lengths)
                strength = np.maximum(0, (strength - 0.27) / 0.73) ** 0.7
            strength[self.exact.get(stem(token), [])] = 1.0
            self.cache[token] = strength
        return self.cache[token]

    def matrix(self, tokens):
        return np.stack([self.match(token) for token in tokens]) if tokens else np.zeros((1, self.size), dtype=np.float32)


def best_two(values):
    vals = np.sort(values)
    return (float(vals[-1]), float(vals[-2]) if len(vals) > 1 else 0.0)


def evidence_windows(matches, size):
    if len(matches) <= size:
        return matches.max(axis=0, keepdims=True)
    windows = np.lib.stride_tricks.sliding_window_view(matches, size, axis=0)[::4]
    return np.vstack([
        windows.max(axis=2), matches[-size:].max(axis=0),
    ])


def evidence_features(rows, corpus):
    idf, unseen, _ = corpus.body
    columns = (
        "ev_body_soft", "ev_title_soft", "ev_union_soft", "ev_min_body", "ev_min_union", "ev_tail_core",
        "ev_subject_body", "ev_relation_body", "ev_relation_union", "ev_subject_title", "ev_relation_title",
        "ev_short_match", "ev_short_joint", "ev_short_distinct", "ev_long_match", "ev_long_joint",
        "ev_sentence_match", "ev_second_sentence", "ev_sentence_joint", "ev_sentence_title_context",
        "ev_first_match", "ev_sentence_density", "ev_answer_match", "ev_answer_joint", "ev_answer_relation",
        "ev_intent_hits", "ev_intent_title", "ev_intent_joint", "ev_intent_with_answer", "ev_intent_missing",
        "ev_answer_sentence_ratio", "ev_answer_specific_unit", "ev_life_date", "ev_question_numeric_match",
        "ev_earliest_context_year", "ev_latest_context_year", "ev_answer_in_definition", "ev_first_title_link",
        "ev_title_topic_mismatch", "ev_value_shared", "ev_value_votes", "ev_value_other_article", "ev_value_own",
    )
    results = np.zeros((len(rows.data), len(columns)), dtype=np.float32)
    for group in rows.data.groupby("query_id", sort=False).indices.values():
        first = group[0]
        qwords = tuple(dict.fromkeys(w for w in rows.query_tokens[first] if w not in STOP)) or rows.query_tokens[first] or ("?",)
        qstems = tuple(map(stem, qwords))
        aligner = SoftWordMatches(qwords)
        base_weight = np.array([idf.get(s, unseen) for s in qstems], dtype=np.float32)
        base_weight /= base_weight.sum()
        names = set(named_tokens(rows.queries.iat[first]))
        title_matches = np.stack([aligner.matrix(rows.title_tokens[i]).max(axis=0) for i in group])
        title_common = title_matches.mean(axis=0)
        topic = np.maximum(title_common, [0.85 if s in names else 0.0 for s in qstems])
        subject_weight = base_weight * (0.12 + topic)
        subject_weight /= subject_weight.sum()
        relation_weight = base_weight * (1.1 - topic)
        relation_weight /= relation_weight.sum()
        query = rows.queries.iat[first]
        qtype = rows.question_type.iat[first]
        cues = tuple(d for q, d in INTENT_RULES if q.search(query))
        anchor = ANSWER_PATTERNS.get(PATTERN_WORDS.get(qtype, ""))
        qvalues = set(VALUE_PATTERN.findall(query))
        candidate_votes = []
        for i, title_alignment in zip(group, title_matches):
            text = rows.text.iat[i]
            title = rows.titles.iat[i]
            segments = [(s, words(s)) for s in SENTENCE.split(text) if s.strip()]
            segments = [(s, toks) for s, toks in segments if toks] or [("", ())]
            all_tokens = tuple(t for _, toks in segments for t in toks)
            mat = aligner.matrix(all_tokens)
            starts = np.r_[0, np.cumsum([len(toks) for _, toks in segments])[:-1]]
            sent = np.maximum.reduceat(mat, starts, axis=0)
            body = mat.max(axis=0)
            union = np.maximum(body, title_alignment)
            short, long = evidence_windows(mat, 15), evidence_windows(mat, 33)
            short_s, short_r = short @ subject_weight, short @ relation_weight
            long_s, long_r = long @ subject_weight, long @ relation_weight
            sent_s, sent_r, sent_main = sent @ subject_weight, sent @ relation_weight, sent @ base_weight
            joint = np.sqrt(sent_s * sent_r)
            has_anchor = np.array([bool(anchor and anchor.search(s)) for s, _ in segments])
            cue_counts = np.array([sum(bool(p.search(s)) for p in cues) for s, _ in segments])
            has_cue = cue_counts > 0
            cued_anchor = has_anchor & has_cue
            near_number = [s for (s, _), z in zip(segments, joint) if z >= max(0.28, 0.70 * joint.max())]
            qnum_pattern = ANSWER_PATTERNS["year"] if qtype == "date" else VALUE_PATTERN
            votes = {}
            if qtype in ("date", "amount"):
                for j, (s, _) in enumerate(segments):
                    weight = float(joint[j] * (0.45 + 0.55 * (has_cue[j] or not cues)))
                    if weight >= 0.23:
                        for value in qnum_pattern.findall(s):
                            if value not in qvalues:
                                votes[value] = max(votes.get(value, 0.0), weight)
            candidate_votes.append(votes)
            context_years = [int(v) for s in near_number for v in ANSWER_PATTERNS["year"].findall(s) if len(v) == 4]
            first_title = set(map(stem, segments[0][1][:10]))
            title_core = {stem(w) for w in rows.title_tokens[i] if w not in STOP}
            title_link = len(title_core & first_title) / max(1, len(title_core))
            first_definition = bool(re.search(r"\s[—–−]\s", segments[0][0][:140]))
            top, second = best_two(sent_main)
            result = (
                body @ base_weight, title_alignment @ base_weight, union @ base_weight, body.min(), union.min(),
                np.mean(np.sort(body)[:max(1, len(body) // 2)]), body @ subject_weight, body @ relation_weight,
                union @ relation_weight, title_alignment @ subject_weight, title_alignment @ relation_weight,
                (short @ base_weight).max(), np.sqrt(short_s * short_r).max(), (short >= 0.55).mean(axis=1).max(),
                (long @ base_weight).max(), np.sqrt(long_s * long_r).max(), top, second, joint.max(),
                (np.maximum(sent, title_alignment) @ base_weight).max(), sent_main[0],
                np.max(sent_main / np.sqrt([max(6, len(toks)) for _, toks in segments])),
                sent_main[has_anchor].max(initial=0), joint[has_anchor].max(initial=0), sent_r[has_anchor].max(initial=0),
                int(has_cue.sum()), sum(bool(p.search(title)) for p in cues), joint[has_cue].max(initial=0),
                joint[cued_anchor].max(initial=0), bool(cues) and not has_cue.any(), has_anchor.mean(),
                any(ANSWER_UNITS.search(s) and VALUE_PATTERN.search(s) for s in near_number), bool(LIFE_DATES.search(text)),
                len(qvalues.intersection(VALUE_PATTERN.findall(text))), min(context_years, default=np.nan),
                max(context_years, default=np.nan), (joint[0] * title_link) if first_definition else 0, title_link,
                float(body @ subject_weight - title_alignment @ subject_weight), 0.0, 0.0, 0.0, 0.0,
            )
            results[i] = result
        article_keys = rows.a_keys.iloc[group].tolist()
        for a, (i, votes) in enumerate(zip(group, candidate_votes)):
            best_vote, best_count, best_cross = 0.0, 0, 0.0
            for value, own in votes.items():
                other = [(b, x[value]) for b, x in enumerate(candidate_votes) if b != a and value in x]
                best_vote = max(best_vote, own * sum(v for _, v in other))
                best_count = max(best_count, len(other))
                best_cross = max(best_cross, own * sum(v for b, v in other if article_keys[b] != article_keys[a]))
            results[i, -4:] = (best_vote, best_count, best_cross, max(votes.values(), default=0.0))
    features = pd.DataFrame(results, columns=columns)
    rank_columns = ("ev_union_soft", "ev_short_joint", "ev_sentence_joint", "ev_answer_joint", "ev_intent_joint",
                    "ev_intent_with_answer", "ev_earliest_context_year", "ev_value_shared", "ev_relation_union")
    for name in rank_columns:
        grouped = features[name].groupby(rows.q_keys, sort=False)
        features[f"rank_{name}"] = grouped.rank(method="average", pct=True).astype(np.float32)
        features[f"gap_{name}"] = grouped.transform("max") - features[name]
    return features

In [ ]:
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.feature_extraction.text import TfidfVectorizer



def query_components(frame):
    q, queries = pd.factorize(frame.query_id, sort=True)
    d, docs = pd.factorize(frame.document_id, sort=True)
    size = len(queries) + len(docs)
    graph = coo_matrix((np.ones(len(frame), dtype=np.int8), (q, d + len(queries))), shape=(size, size))
    _, labels = connected_components(graph.tocsr(), directed=False)
    return labels[q]


def core_terms(raw, stems):
    return frozenset(s for word, s in zip(raw, stems) if word not in STOP and len(word) >= 2)


class LabeledNeighbors:
    """Поиск похожих пар в размеченной части с исключением текущей группы."""

    def __init__(self, rows):
        self.components = query_components(rows.data)
        self.query_positions = np.flatnonzero(~rows.q_keys.duplicated().to_numpy())
        self.query_to_id = {rows.q_keys.iat[i]: j for j, i in enumerate(self.query_positions)}
        self.doc_query = rows.q_keys.map(self.query_to_id).to_numpy()
        self.labels = rows.data.relevance.to_numpy()
        self.base_rate = float(self.labels.mean())
        self.query_rows = list(rows.data.groupby("query_id", sort=False).indices.values())
        self.by_article = rows.data.groupby("article_id", sort=False).indices
        self.by_title = defaultdict(list)
        self.titles = [core_terms(w, s) for w, s in zip(rows.title_tokens, rows.title_stems)]
        self.bodies = [core_terms(w, s) for w, s in zip(rows.text_tokens, rows.text_stems)]
        for i, title in enumerate(self.titles):
            if title:
                self.by_title[title].append(i)
        self.word_vectorizer = TfidfVectorizer(token_pattern=r"[^\s]+", ngram_range=(1, 2), sublinear_tf=True,
                                                dtype=np.float32, max_features=25_000)
        self.char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True,
                                                dtype=np.float32, max_features=30_000)
        stem_text, raw_text = self.query_text(rows, self.query_positions)
        self.word_matrix = self.word_vectorizer.fit_transform(stem_text)
        self.char_matrix = self.char_vectorizer.fit_transform(raw_text)

    @staticmethod
    def query_text(rows, positions):
        raw, stems = [], []
        for i in positions:
            terms = [(w, s) for w, s in zip(rows.query_tokens[i], rows.query_stems[i]) if w not in STOP]
            if not terms:
                terms = list(zip(rows.query_tokens[i], rows.query_stems[i])) or [("пусто", "пусто")]
            raw.append(" ".join(w for w, _ in terms))
            stems.append(" ".join(s for _, s in terms))
        return stems, raw

    def features(self, rows, training=False):
        positions = np.flatnonzero(~rows.q_keys.duplicated().to_numpy())
        stem_text, raw_text = self.query_text(rows, positions)
        similarities = (self.word_vectorizer.transform(stem_text) @ self.word_matrix.T).toarray() * 0.68
        similarities += (self.char_vectorizer.transform(raw_text) @ self.char_matrix.T).toarray() * 0.32
        names = (
            "memory_seen_article", "memory_seen_title", "memory_article_prior", "memory_article_sim_pos", "memory_article_sim_neg",
            "memory_article_qrate", "memory_article_text_pos", "memory_article_text_neg", "memory_nearest_query",
            "memory_pair_pos", "memory_pair_neg", "memory_pair_title_pos", "memory_pair_title_neg",
            "memory_pair_vote", "memory_pair_evidence", "memory_pair_rank_diff",
        )
        output = np.zeros((len(rows.data), len(names)), dtype=np.float32)
        for j, indices in enumerate(rows.data.groupby("query_id", sort=False).indices.values()):
            sim = similarities[j]
            banned = None
            if training:
                banned = self.components == self.components[indices[0]]
                sim[self.components[self.query_positions] == self.components[indices[0]]] = -1
            num = min(9, len(sim))
            nearest = np.argpartition(sim, len(sim) - num)[-num:]
            nearest = sorted((int(k) for k in nearest if sim[k] >= 0.16), key=lambda k: (-sim[k], k))
            near_docs = [i for k in nearest for i in self.query_rows[k]]
            for idx in indices:
                title = core_terms(rows.title_tokens[idx], rows.title_stems[idx])
                body = core_terms(rows.text_tokens[idx], rows.text_stems[idx])
                same_article = self.by_article.get(rows.a_keys.iat[idx], ())
                same_title = self.by_title.get(title, ())
                article_docs = sorted(set(same_article) | set(same_title))
                if banned is not None:
                    article_docs = [i for i in article_docs if not banned[i]]
                article_strength = [0., 0.]
                article_body = [0., 0.]
                hit = sum(self.labels[i] for i in article_docs)
                article_w = 0.0
                article_v = 0.0
                for i in article_docs:
                    s = max(0., float(sim[self.doc_query[i]]))
                    l = self.labels[i]
                    lexical = len(body & self.bodies[i]) / max(1, len(body | self.bodies[i]))
                    article_strength[l] = max(article_strength[l], s)
                    article_body[l] = max(article_body[l], s * lexical)
                    article_w += s ** 2
                    article_v += s ** 2 * l
                best_pair = [0., 0.]
                best_title = [0., 0.]
                pair_w = 0.0
                pair_v = 0.0
                for i in near_docs:
                    s = float(sim[self.doc_query[i]])
                    lexical = len(body & self.bodies[i]) / max(1, len(body | self.bodies[i]))
                    title_match = len(title & self.titles[i]) / max(1, len(title | self.titles[i]))
                    l = self.labels[i]
                    joint = s * (0.7 * lexical + 0.3 * title_match)
                    best_pair[l] = max(best_pair[l], joint)
                    best_title[l] = max(best_title[l], s * title_match)
                    pair_w += joint ** 3
                    pair_v += joint ** 3 * l
                output[idx] = (
                    np.log1p(sum(banned is None or not banned[i] for i in same_article)),
                    np.log1p(sum(banned is None or not banned[i] for i in same_title)),
                    (hit + 5 * self.base_rate) / (len(article_docs) + 5), article_strength[1], article_strength[0],
                    (article_v + 0.35 * self.base_rate) / (article_w + 0.35), article_body[1], article_body[0],
                    max(0., float(sim.max())), best_pair[1], best_pair[0], best_title[1], best_title[0],
                    (pair_v + 0.035 * self.base_rate) / (pair_w + 0.035), pair_w, best_pair[1] - best_pair[0],
                )
        return pd.DataFrame(output, columns=names)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GroupKFold



def learned_text_fields(rows, corpus):
    idf, unseen, _ = corpus.body
    q_fields, body_fields, context_fields, title_fields = [], [], [], []
    for i, (q_raw, q_stem, body, title) in enumerate(zip(rows.query_tokens, rows.query_stems, rows.text_stems, rows.title_stems)):
        typ = rows.question_type.iat[i]
        main = tuple(dict.fromkeys(s for w, s in zip(q_raw, q_stem) if w not in STOP and not w.isdigit()))
        intents = [f"intent_{j}" for j, (pattern, _) in enumerate(INTENT_RULES) if pattern.search(rows.queries.iat[i])]
        q_tags = [f"type_{typ}", *intents]
        if q_stem:
            q_tags.append(f"first_{q_stem[0]}")
        if len(q_stem) > 1:
            q_tags.append(f"prefix_{q_stem[0]}_{q_stem[1]}")
        q_tags.extend(f"word_{w}" for w in main)
        q_tags.extend(f"type_{typ}_{w}" for w in main)
        q_fields.append(" ".join(q_tags))
        q_set = set(main)
        anchor = ANSWER_PATTERNS.get(PATTERN_WORDS.get(typ, ""))
        sections = []
        for part in SENTENCE.split(rows.text.iat[i]):
            if part.strip():
                tokens = tuple(map(stem, words(part)))
                hit = sorted(q_set.intersection(tokens))
                score = sum(idf.get(t, unseen) for t in hit) / np.sqrt(max(8, len(tokens)))
                score += 0.5 * bool(anchor and anchor.search(part))
                sections.append((score, tokens))
        sections.sort(key=lambda t: -t[0])
        focus = tuple(t for _, segment in sections[:2] for t in segment)
        body_fields.append(encode_field(body))
        context_fields.append(encode_field(focus))
        title_fields.append(encode_field(title))
    return q_fields, (body_fields, context_fields, title_fields)


def encode_field(tokens):
    generic = ["num_year" if w.isdigit() and len(w) == 4 else "num_other" if w.isdigit() else w for w in tokens]
    terms = [w for w in generic if len(w) >= 2]
    phrases = [f"p_{a}_{b}" for a, b in zip(generic, generic[1:]) if a not in STOP and b not in STOP]
    return " ".join(terms + phrases)


class LearnedLexicon:
    """Контраст частот слов в релевантных и нерелевантных кандидатах."""

    def __init__(self, train_rows, test_rows, corpus):
        self.train = train_rows.data
        self.test = test_rows.data
        train_q, train_fields = learned_text_fields(train_rows, corpus)
        test_q, test_fields = learned_text_fields(test_rows, corpus)
        q_vectorizer = CountVectorizer(token_pattern=r"[^\s]+", lowercase=False, binary=True, dtype=np.float32,
                                       min_df=12, max_features=2_200)
        self.q_train = q_vectorizer.fit_transform(train_q)
        self.q_test = q_vectorizer.transform(test_q)
        self.field_matrices = []
        for j, (train_texts, test_texts) in enumerate(zip(train_fields, test_fields)):
            field = CountVectorizer(token_pattern=r"[^\s]+", lowercase=False, binary=True, dtype=np.float32,
                                    min_df=6 if j < 2 else 4, max_features=(10_000, 7_000, 6_000)[j])
            self.field_matrices.append((field.fit_transform(train_texts), field.transform(test_texts)))

    def score(self, fit_indices, score_indices=None):
        train = self.train.iloc[fit_indices]
        y = train.relevance.to_numpy()
        q_size = train.groupby("query_id", sort=False).relevance.transform("size").to_numpy()
        q_pos = train.groupby("query_id", sort=False).relevance.transform("sum").to_numpy()
        w_pos = (y / np.maximum(1, q_pos)).astype(np.float32)
        w_neg = ((1 - y) / np.maximum(1, q_size - q_pos)).astype(np.float32)
        train_q = self.q_train[fit_indices]
        q_pos_count = np.asarray(train_q.T @ w_pos).ravel()
        q_neg_count = np.asarray(train_q.T @ w_neg).ravel()
        reliability = (q_pos_count + q_neg_count) / (q_pos_count + q_neg_count + 12)
        if score_indices is None:
            q_index = self.test.query_id
            q_transform = self.q_test
        else:
            q_index = self.train.query_id.iloc[score_indices]
            q_transform = self.q_train[score_indices]
        q_codes, _ = pd.factorize(q_index, sort=False)
        q_unique = q_transform[~q_index.duplicated().to_numpy()]
        q_norm = np.asarray(q_unique @ reliability).clip(1.0)[:, None]
        parts = []
        for train_all, test_all in self.field_matrices:
            x = train_all[fit_indices]
            score_matrix = test_all if score_indices is None else train_all[score_indices]
            p_prior = np.asarray(x.T @ w_pos).ravel()
            n_prior = np.asarray(x.T @ w_neg).ravel()
            p_prior = (p_prior + 0.6) / (w_pos.sum() + 3)
            n_prior = (n_prior + 0.6) / (w_neg.sum() + 3)
            global_score = np.log(p_prior / n_prior)
            positive = (train_q.T @ x.multiply(w_pos[:, None])).toarray()
            negative = (train_q.T @ x.multiply(w_neg[:, None])).toarray()
            positive += 9 * p_prior[None, :]
            negative += 9 * n_prior[None, :]
            positive /= q_pos_count[:, None] + 9
            negative /= q_neg_count[:, None] + 9
            np.log(positive, out=positive)
            np.log(negative, out=negative)
            positive -= negative
            positive -= global_score[None, :]
            positive *= reliability[:, None]
            profile = (q_unique @ positive) / q_norm
            out = np.zeros((score_matrix.shape[0], 7), dtype=np.float32)
            for i, code in enumerate(q_codes):
                j = score_matrix.indices[score_matrix.indptr[i]:score_matrix.indptr[i + 1]]
                if len(j) == 0:
                    continue
                s = profile[code, j]
                k = min(4, len(s))
                p = np.partition(s, len(s) - k)
                gl = global_score[j]
                out[i] = (s.mean(), s.max(), p[-k:].mean(), s.min(), gl.mean(), gl.max(), (s + 0.4 * gl).max())
            parts.append(out)
        return np.column_stack(parts)

    def matrices(self):
        columns = [f"lexicon_{field}_{stat}" for field in ("body", "context", "title") for stat in (
            "mean", "max", "top", "min", "prior_mean", "prior_max", "combined_max")]
        out = np.zeros((len(self.train), len(columns)), dtype=np.float32)
        components = query_components(self.train)
        for fit, val in GroupKFold(n_splits=4).split(self.train, groups=components):
            out[val] = self.score(fit, val)
        test = self.score(np.arange(len(self.train)))
        return pd.DataFrame(out, columns=columns), pd.DataFrame(test, columns=columns)

In [ ]:
from collections import defaultdict
import re

import numpy as np
import pandas as pd



NUMERIC_SPAN = re.compile(r"(?<!\w)(?:\d{1,3}(?:[ \u00a0\u202f]\d{3})+|\d+)(?:[.,]\d+)?(?!\w)")
SPELL_NUMBER = re.compile(r"\b(?:один|одна|два|две|три|четыре|пять|шесть|семь|восемь|девять|десять|двенадцать|двадцать|тридцать|сто|сотня|несколько|полтора|половина)\b", re.I)
RUS_MONTH = r"январ\w*|феврал\w*|март\w*|апрел\w*|ма[йея]|июн\w*|июл\w*|август\w*|сентябр\w*|октябр\w*|ноябр\w*|декабр\w*"
DATE_SPAN = re.compile(r"\b(?:(?:\d{1,2}\s+(?:" + RUS_MONTH + r")\s+)?[12]\d{3}|[IVXLCDM]{1,8}\s+(?:век\w*|в\.)|\d{1,2}\s+(?:" + RUS_MONTH + r"))\b", re.I)
LIFESPAN = re.compile(r"\(([^()]{0,100})\s+[—–−]\s+([^()]{0,100})\)")
NAME_SPAN = re.compile(r"(?<![\w])(?:[А-ЯЁA-Z]\.\s*){0,2}[А-ЯЁA-Z][а-яёa-z]+(?:[-–][А-ЯЁA-Z]?[а-яёa-z]+)?(?:\s+[А-ЯЁA-Z][а-яёa-z]+(?:[-–][А-ЯЁA-Z]?[а-яёa-z]+)?){0,2}")
QUOTE_SPAN = re.compile(r"«([^»]{1,80})»|[\"„“]([^\"“”]{1,80})[\"“”]")
LOCATION_CUE = re.compile(r"\b(?:в|на|из|около|возле|под|между|близ|окрестност\w*|город\w*|стран\w*|регион\w*)\s*$", re.I)
DEFINITION_CUE = re.compile(r"\b(?:называ\w*|извест\w*|под\s+названи\w*|обознач\w*|считает\w*|представляет собой)\b|[—–:]", re.I)
UNIT_RULES = (
    (r"\b(?:лет|возраст\w*|продолжительность\s+жизни)\b", r"\b(?:лет|год\w*|годич\w*|дожива\w*|возраст\w*)\b"),
    (r"\b(?:дней|месяц\w*|час\w*|длит\w*|длилась|длился|времени)\b", r"\b(?:дн\w*|сут\w*|месяц\w*|час\w*|минут\w*|недел\w*|лет|год\w*)\b"),
    (r"\b(?:вес\w*|весит|масс\w*|кг|килограмм\w*)\b", r"\b(?:кг|килограмм\w*|грамм\w*|тонн\w*|вес\w*|масс\w*)\b"),
    (r"\b(?:длин\w*|глубин\w*|высот\w*|рост|расстояни\w*|ширин\w*|протяжённост\w*)\b", r"\b(?:см|км|м|метр\w*|километр\w*|фут\w*|дюйм\w*)\b"),
    (r"\b(?:площад\w*|территори\w*)\b", r"\b(?:га|км²|км2|гектар\w*|кв\.|квадратн\w*)\b"),
    (r"\b(?:серий|сезон\w*|эпизод\w*|частей|фильм\w*|книг\w*)\b", r"\b(?:серий|сери[яи]|сезон\w*|эпизод\w*|част\w*|фильм\w*|книг\w*)\b"),
    (r"\b(?:населени\w*|человек|людей|жител\w*)\b", r"\b(?:человек|чел\.|жител\w*|тысяч\w*|миллион\w*|млн|тыс)\b"),
    (r"\b(?:стоит|цен\w*|стоимост\w*|зарплат\w*|бюджет\w*)\b", r"\b(?:руб\w*|доллар\w*|евро|фунт\w*|ден\w*|тыс\.?|млн\.?)\b|[$₽€]"),
    (r"\b(?:процент\w*|дол[яюи]|часть)\b|%", r"%|\bпроцент\w*\b"),
)
UNIT_RULES = tuple((re.compile(a, re.I), re.compile(b, re.I)) for a, b in UNIT_RULES)
ANS_SPAN_COLUMNS = (
    "span_count", "span_distinct", "span_support", "span_relation", "span_subject", "span_trigger", "span_union_title",
    "span_short", "span_tight", "span_order", "span_value_quality", "span_score_mean", "span_score_second",
    "span_best_alternate", "span_life_date", "span_num_match_unit", "span_doc_intro", "span_title_identity",
    "span_support_vote", "span_support_other_articles", "span_value_overlap", "span_own_gap", "span_typed_logpos",
)


def normalized_span(text):
    if any(c.isdigit() for c in text):
        return "".join(c for c in text if c.isdigit() or c in ".,").replace(",", ".")
    return " ".join(stem(w) for w in words(text) if w not in STOP)


def proposed_spans(text, typ, query, unit_patterns):
    if typ in ("date", "amount"):
        spans = list(DATE_SPAN.finditer(text)) if typ == "date" else list(NUMERIC_SPAN.finditer(text)) + list(SPELL_NUMBER.finditer(text))
        for m in spans:
            vicinity = text[max(0, m.start() - 15):m.end() + 34]
            unit = max((bool(p.search(vicinity)) for p in unit_patterns), default=False)
            quality = 1.0 if typ == "date" else (.70 + .30 * unit if unit_patterns else 0.88)
            yield m.start(), m.end(), normalized_span(m.group()), quality, unit
    else:
        if typ in ("name", "choice", "definition", "other"):
            for m in QUOTE_SPAN.finditer(text):
                yield m.start(), m.end(), normalized_span(m.group()), 0.95, False
        if typ in ("person", "place", "choice", "name", "other", "yesno", "reason"):
            for m in NAME_SPAN.finditer(text):
                geo = bool(LOCATION_CUE.search(text[max(0, m.start() - 28):m.start()]))
                before = text[max(0, m.start() - 45):m.start()]
                multi = len(words(m.group())) > 1
                definition = bool(DEFINITION_CUE.search(before))
                quality = .64 + (.25 * geo if typ == "place" else .22 * multi if typ == "person" else .23 * definition)
                yield m.start(), m.end(), normalized_span(m.group()), quality, False


def answer_span_features(rows, corpus):
    idf, unseen, _ = corpus.body
    result = np.zeros((len(rows.data), len(ANS_SPAN_COLUMNS)), dtype=np.float32)
    for group in rows.data.groupby("query_id", sort=False).indices.values():
        first = group[0]
        q = rows.queries.iat[first]
        qtype = rows.question_type.iat[first]
        qraw = tuple(dict.fromkeys(t for t in rows.query_tokens[first] if t not in STOP)) or ("?",)
        qstems = tuple(map(stem, qraw))
        q_terms = set(qstems)
        existing_values = {normalized_span(m.group()) for m in NUMERIC_SPAN.finditer(q)}
        entities = set(named_tokens(q))
        aligner = SoftWordMatches(qraw)
        title_matches = np.stack([aligner.matrix(rows.title_tokens[i]).max(axis=0) for i in group])
        w = np.asarray([idf.get(t, unseen) for t in qstems], dtype=np.float32)
        w /= w.sum()
        subj_indicator = np.maximum(.85 * np.isin(qstems, list(entities)), title_matches.mean(axis=0))
        subject = w * (.18 + subj_indicator)
        subject /= subject.sum()
        relation = w * (1.17 - subj_indicator)
        relation /= relation.sum()
        triggers = [d for p, d in INTENT_RULES if p.search(q)]
        units = [unit for expr, unit in UNIT_RULES if expr.search(q)] if qtype == "amount" else []
        article = rows.a_keys.iloc[group].tolist()
        local_values = []
        for j, i in enumerate(group):
            text = rows.text.iat[i]
            word_matches = list(WORD.finditer(text))
            candidate_tokens = tuple(m.group().lower().replace('ё', 'е') for m in word_matches)
            token_starts = np.array([m.start() for m in word_matches])
            mat = aligner.matrix(candidate_tokens)
            active_triggers = np.array([m.start() for trigger in triggers for m in trigger.finditer(text)])
            life_dates = bool(qtype == "date" and any(DATE_SPAN.search(m[1]) and DATE_SPAN.search(m[2]) for m in LIFESPAN.finditer(text)))
            records = []
            votes = {}
            for start, stop, value, quality, unit in proposed_spans(text, qtype, q, units):
                if not value or value in existing_values or all(word in q_terms for word in value.split()):
                    continue
                left, right = np.searchsorted(token_starts, [start, stop])
                previous = mat[max(0, left-17):left]
                after = mat[right:right+17]
                around = np.maximum(previous.max(axis=0, initial=0), after.max(axis=0, initial=0))
                tight = np.maximum(previous[-6:].max(axis=0, initial=0), after[:6].max(axis=0, initial=0))
                near = float(around @ w)
                strong = float(tight @ w)
                rel, sub = float(around @ relation), float(around @ subject)
                union = float(np.maximum(around, title_matches[j]) @ w)
                dist = np.min(abs(active_triggers - start), initial=500)
                trigger = np.exp(-dist / 60) if triggers else .5
                directional = float(previous[-7:].max(axis=0, initial=0) @ relation)
                proof = quality * (.38 * near + .22 * strong + .22 * union + .10 * trigger + .08 * directional)
                records.append((proof, rel, sub, trigger, union, strong, near, directional, quality, unit, start))
                votes[value] = max(votes.get(value, 0), proof)
            ordered = sorted(records, reverse=True)
            best = ordered[0] if ordered else (0.,) * 11
            result[i, :18] = (
                len(ordered), len(votes), *best[:9], np.mean([r[0] for r in ordered]) if ordered else 0.,
                ordered[1][0] if len(ordered) > 1 else 0.,
                max((r[0] for r in ordered if r[8] < .85), default=0.), life_dates, bool(any(r[9] for r in ordered)),
                bool(re.search(r"\s[—–]\s", text[:140])), float(title_matches[j] @ subject),
            )
            result[i, -1] = np.log1p(best[-1]) if ordered else np.nan
            local_values.append(votes)
        seen = defaultdict(list)
        for j, values in enumerate(local_values):
            for answer, value in values.items():
                if value >= .18:
                    seen[answer].append((j, value))
        for j, i in enumerate(group):
            candidates = []
            for answer, strength in local_values[j].items():
                other = [(a, val) for a, val in seen[answer] if a != j]
                candidates.append((strength * sum(v for _, v in other),
                                   strength * sum(v for a, v in other if article[a] != article[j]), len(other)))
            result[i, -5:-1] = (
                max((a[0] for a in candidates), default=0.), max((a[1] for a in candidates), default=0.),
                max((a[2] for a in candidates), default=0.), result[i, 2] - result[group, 2].max(),
            )
    return pd.DataFrame(result, columns=ANS_SPAN_COLUMNS)

In [ ]:
import numpy as np

import pandas as pd


class ArticleContext:
    """Контекст статьи по доступным фрагментам, без использования разметки."""

    def __init__(self, rows, corpus):
        self.idf, self.unseen, _ = corpus.body
        self.records = self.describe(rows)
        self.articles = {}
        for j in rows.doc_positions:
            self.articles.setdefault(rows.a_keys.iat[j], []).append(self.records[j])

    @staticmethod
    def describe(rows):
        pos, count, char = [normalize_number(rows.data[k]).to_numpy() for k in (
            "passage_index", "article_passage_count", "article_char_count")]
        return [
            (d, p, n, length, set(t) - STOP, set(b) - STOP, len(text))
            for d, p, n, length, t, b, text in zip(rows.d_keys, pos, count, char, rows.title_stems, rows.text_stems, rows.text)
        ]

    @staticmethod
    def rescale(length, count):
        if not np.isfinite(length) or length < 1 or not np.isfinite(count) or count < 1:
            return length
        choices = np.array([length, length / 10, length / 100, length / 1000])
        costs = abs(np.log((choices / count) / 540)) + np.array([0., .75, 1.1, 1.35])
        return choices[np.argmin(costs)]

    def features(self, rows):
        records = self.describe(rows)
        labels = ("neighbor_docs", "observed_before", "observed_after", "article_vocab_match", "article_title_match",
                  "article_max_match", "article_mean_match", "article_first_match", "article_closest_match", "article_match_gain",
                  "left_passage_match", "right_passage_match", "known_body_share", "article_title_support",
                  "meta_count", "meta_char_clean", "meta_avg_passage", "meta_len_relative", "meta_closest_index", "meta_corrected")
        out = np.zeros((len(rows.data), len(labels)), dtype=np.float32)
        for group in rows.data.groupby("query_id", sort=False).indices.values():
            first = group[0]
            q = tuple(dict.fromkeys(s for w, s in zip(rows.query_tokens[first], rows.query_stems[first]) if w not in STOP)) or ("",)
            weights = np.asarray([self.idf.get(t, self.unseen) for t in q])
            weights /= weights.sum()
            local = {}
            for i in group:
                local.setdefault(rows.a_keys.iat[i], []).append(records[i])
            for i in group:
                doc, pos, count, length, title, body, text_len = records[i]
                a = rows.a_keys.iat[i]
                observed = {r[0]: r for r in self.articles.get(a, []) + local[a] if r[0] != doc}
                related = list(observed.values())
                related.sort(key=lambda r: (not np.isfinite(r[1]), abs(r[1]-pos) if np.isfinite(r[1]) and np.isfinite(pos) else 0))
                related = related[:50]
                def coverage(tokens):
                    return sum(w for term, w in zip(q, weights) if term in tokens)

                others = set().union(*(r[5] for r in related))
                headings = set().union(*(r[4] for r in related))
                strength = [coverage(r[5]) for r in related]
                before = [s for s, r in zip(strength, related) if np.isfinite(pos) and pos-5 <= r[1] < pos]
                after = [s for s, r in zip(strength, related) if np.isfinite(pos) and pos < r[1] <= pos+5]
                first_passage = min(related, key=lambda r: r[1] if np.isfinite(r[1]) else np.inf, default=None)
                count_consensus = [r[2] for r in related if np.isfinite(r[2]) and r[2] >= 1]
                if np.isfinite(count) and count >= 1:
                    count_consensus.append(count)
                c = np.median(count_consensus) if count_consensus else np.nan
                known_chars = [self.rescale(r[3], r[2] if r[2] >= 1 else c) for r in related if np.isfinite(r[3]) and r[3] >= 1]
                adjusted = self.rescale(length, c)
                if np.isfinite(adjusted) and adjusted >= 1:
                    known_chars.append(adjusted)
                l = np.median(known_chars) if known_chars else np.nan
                ratio = l/c if c >= 1 else np.nan
                delta = [abs(r[1]-pos) for r in related if np.isfinite(r[1]) and np.isfinite(pos)]
                out[i] = (
                    np.log1p(len(related)), len(before), len(after), coverage(others | body), coverage(headings | title),
                    max(strength, default=0.), np.mean(strength) if strength else 0.,
                    coverage(first_passage[5]) if first_passage else 0., strength[0] if strength else 0.,
                    coverage(others | body) - coverage(body), max(before, default=0.), max(after, default=0.),
                    len(body & others) / max(1, len(body)), len(title & headings) / max(1, len(title)),
                    np.log1p(c), np.log1p(l), np.log1p(ratio), np.log1p(text_len/ratio) if ratio > 0 else np.nan,
                    np.log1p(min(delta)) if delta else np.nan, float(np.isfinite(length) and adjusted < length),
                )
        return pd.DataFrame(out, columns=["xarticle_" + name for name in labels])

In [ ]:
import numpy as np
import pandas as pd



def neighborhood_features(rows, corpus):
    """Псевдообратная связь и распространение скоров между статьями выдачи."""
    idf, unseen, _ = corpus.body
    out = np.empty((len(rows.data), 32), dtype=np.float32)
    stop = {stem(t) for t in STOP}
    for group in rows.data.groupby("query_id", sort=False).indices.values():
        n = len(group)
        query = {t for r, t in zip(rows.query_tokens[group[0]], rows.query_stems[group[0]]) if r not in STOP}
        q_weights = {t: idf.get(t, unseen) for t in query}
        q_total = max(sum(q_weights.values()), 1.)
        titles = [set(rows.title_stems[i]) for i in group]
        bodies = [set(rows.text_stems[i]) for i in group]
        intro = [set(rows.text_stems[i][:45]) for i in group]
        articles = rows.a_keys.iloc[group].to_numpy()
        separate = articles[:, None] != articles[None, :]
        within = ~separate & ~np.eye(n, dtype=bool)

        def cover(terms):
            return sum(q_weights[t] for t in terms & query) / q_total

        body, title, head = (np.array([cover(t) for t in field]) for field in (bodies, titles, intro))
        united = np.array([cover(t | b) for t, b in zip(titles, bodies)])
        seed = .4 * title + .25 * body + .2 * united + .15 * head
        rep = np.array([(articles == a).sum() for a in articles])
        strength = np.exp(5 * (seed - seed.max())) / rep
        vocab = sorted(set.union(*bodies, *titles) - query - stop)
        by_term = {t: j for j, t in enumerate(vocab)}
        weights = np.array([min(idf.get(t, unseen), 8.) ** .7 for t in vocab])
        matrix = np.zeros((n, len(vocab)))
        for j, (b, t, h) in enumerate(zip(bodies, titles, intro)):
            for terms, weight in ((b, 1.), (t, 2.5), (h, .6)):
                cols = [by_term[k] for k in terms if k in by_term]
                matrix[j, cols] += weight
        matrix *= weights
        matrix /= np.linalg.norm(matrix, axis=1, keepdims=True).clip(1e-10)
        sim = matrix @ matrix.T
        np.fill_diagonal(sim, 0.)
        shared = sim * separate
        connection = shared**2
        transition = connection / connection.sum(axis=1, keepdims=True).clip(1e-10)
        walk = strength / strength.sum()
        for _ in range(4):
            walk = .55 * strength / strength.sum() + .45 * (walk @ transition)
        by_query = np.stack((body, title, head, united, seed), axis=1)
        article_max = np.stack([by_query[articles == a].max(axis=0) for a in articles])
        other_body = (shared @ body) / shared.sum(axis=1).clip(1e-10)
        other_title = (shared @ title) / shared.sum(axis=1).clip(1e-10)
        high = shared * strength[None, :]
        out[group] = np.column_stack((by_query, article_max, article_max - by_query,
            shared.max(axis=1), shared.sum(axis=1), np.sort(shared, axis=1)[:, -min(2,n):].mean(axis=1),
            high.max(axis=1), high.sum(axis=1), (sim * within).max(axis=1),
            (sim * within).sum(axis=1), other_body, other_title, walk, walk - strength / strength.sum(),
            (shared * (seed[None, :] >= .5)).sum(axis=1), (shared * (seed[None, :] < .5)).sum(axis=1),
            (shared * (title[None, :] > .75)).max(axis=1), (shared * (body[None, :] > .75)).max(axis=1),
            np.sum(shared > .16, axis=1), np.sum((sim * within) > .16, axis=1)))
    return pd.DataFrame(out, columns=[f"feedback_{i}" for i in range(out.shape[1])])

## CatBoost

Обучаем CatBoostRanker с QuerySoftMax. Кандидаты одного запроса образуют группу.


In [ ]:
from catboost import CatBoostRanker, Pool


def fit_catboost(train, design, test_design):
    order = np.argsort(train.query_id.to_numpy(), kind="stable")
    pool = Pool(
        design.iloc[order],
        label=train.relevance.to_numpy()[order],
        group_id=train.query_id.to_numpy()[order],
    )
    model = CatBoostRanker(
        loss_function="QuerySoftMax",
        iterations=600,
        depth=6,
        learning_rate=0.05,
        random_seed=20261107,
        thread_count=3,
        task_type="CPU",
        allow_writing_files=False,
        verbose=False,
    )
    model.fit(pool)
    return model.predict(test_design)


def classical_scores(train, test):
    with threadpool_limits(limits=2, user_api="blas"), threadpool_limits(limits=4, user_api="openmp"):
        rows_pair = PreparedRows(train), PreparedRows(test)
        training_rows, test_rows = rows_pair
        corpus = CorpusStats(training_rows)
        characters = CharacterSimilarity(training_rows)
        semantic = DistributionalSimilarity(training_rows, corpus)
        article_context = ArticleContext(training_rows, corpus)
        base = tuple(full_features(rows, corpus, characters, semantic) for rows in rows_pair)
        evidence = tuple(evidence_features(rows, corpus) for rows in rows_pair)
        memory = LabeledNeighbors(training_rows)
        neighbors = memory.features(training_rows, training=True), memory.features(test_rows)
        lexicon = LearnedLexicon(training_rows, test_rows, corpus).matrices()
        local = tuple(pd.concat([answer_span_features(rows, corpus), article_context.features(rows),
                                 neighborhood_features(rows, corpus)], axis=1) for rows in rows_pair)
        design, test_design = (pd.concat(items, axis=1) for items in zip(base, evidence, neighbors, lexicon, local))
        assert design.columns.is_unique and design.columns.equals(test_design.columns)
        print(f"Готовы {design.shape[1]} признаков для {len(train):,} обучающих и {len(test):,} оцениваемых пар", flush=True)
        return fit_catboost(train, design, test_design)


## Оценка Jina

Jina получает запрос и фрагмент с заголовком. При первом запуске загружаем готовые веса и токенизатор, затем сверяем SHA-256 обоих файлов.

In [ ]:
"""Инференс опубликованного Jina Reranker v2 в ONNX Runtime."""
from hashlib import sha256
from html import unescape
import os
import re
from pathlib import Path
import tempfile
import unicodedata
from urllib.request import urlopen

import numpy as np
import pandas as pd
import onnxruntime as ort
from tokenizers import Tokenizer


REPAIR_MARK = re.compile(r"(?:[РС][А-Яа-яЁё\u0080-\u00bf]){3,}")
HTML_BLOCK = re.compile(r"</?(?:p|div|br|li|ul|ol|h[1-6])(?:\s+[^<>]*)?/?>", re.I)
HTML_INLINE = re.compile(r"</?[a-z][a-z0-9]*(?:\s+[^<>]*)?/?>", re.I)
NAVIGATION = re.compile(r"^\s*(?:навигация\s*:|главная\s*[|›»>])[^\n]*\n+", re.I)
INVISIBLE = re.compile(r"[\u00ad\u0301\u200b-\u200f\ufeff]")
WIKI_SUFFIX = re.compile(r"\s*[-–—|]\s*википедия\s*$", re.I)

def normalize_passage(value):
    text = str(value) if pd.notna(value) else ""
    if REPAIR_MARK.search(text):
        try:
            fixed = text.encode("cp1251").decode("utf-8")
        except UnicodeError:
            fixed = text
        if len(fixed) < .85 * len(text):
            text = fixed
    text = unicodedata.normalize("NFKC", unescape(unescape(text)))
    text = HTML_BLOCK.sub("\n", text)
    text = HTML_INLINE.sub(" ", text)
    text = INVISIBLE.sub("", text)
    return " ".join(NAVIGATION.sub("", text).split())


def rerank_texts(frame, title_reference=None):
    queries = frame["query"].fillna("").map(normalize_passage).tolist()
    titles = frame["title"].fillna("").map(normalize_passage).str.replace(WIKI_SUFFIX, "", regex=True)
    votes = pd.DataFrame({"article": frame.article_id, "title": titles})
    votes = votes[votes.title.ne("")]
    if title_reference is not None:
        votes = pd.concat((title_reference, votes), ignore_index=True)
    counts = votes.value_counts().rename("count").reset_index()
    lookup = (counts.sort_values(["count", "title"], ascending=[False, True])
              .drop_duplicates("article").set_index("article")["title"])
    titles = titles.where(titles.ne(""), frame.article_id.map(lookup).fillna(""))
    bodies = frame["text"].fillna("").map(normalize_passage)
    passages = [f"{title}\n{text}" if title and not text.casefold().startswith(title.casefold()) else text
                for title, text in zip(titles, bodies)]
    return (queries, passages), votes


def clip_tokens(tokens, limit):
    if len(tokens) <= limit:
        return tokens
    start = int(limit * .68)
    return tokens[:start] + tokens[-(limit-start):]


JINA_REPOSITORY = "jinaai/jina-reranker-v2-base-multilingual"
JINA_REVISION = "9cfeff2df7d40d1b78e75e5e9cebec92a99813c9"
JINA_ARTIFACTS = {
    "tokenizer.json": "3a56def25aa40facc030ea8b0b87f3688e4b3c39eb8b45d5702b3a1300fe2a20",
    "onnx/model_int8.onnx": "c5220cf8fe023f8aa0ed2a3eb787d4451a7f17cf53f6b787e35718dd4b8815c3",
}


def checksum_file(path):
    checksum = sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(4 * 1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()


def jina_checkpoint(directory=None):
    directory = directory or os.environ.get("PMMO_CHECKPOINT_CACHE")
    root = Path(directory) if directory is not None else (
        Path(os.environ.get("XDG_CACHE_HOME", Path.home() / ".cache"))
        / "pmmo-reranking" / JINA_REVISION
    )
    root.mkdir(parents=True, exist_ok=True)
    hosts = (
        f"https://huggingface.co/{JINA_REPOSITORY}/resolve/{JINA_REVISION}",
        f"https://modelscope.cn/models/{JINA_REPOSITORY}/resolve/master",
    )
    paths = []
    for name, digest in JINA_ARTIFACTS.items():
        target = root / Path(name).name
        if not target.is_file() or checksum_file(target) != digest:
            error = None
            for host in hosts:
                temporary_path = None
                try:
                    with urlopen(f"{host}/{name}", timeout=30) as source:
                        with tempfile.NamedTemporaryFile(dir=root, delete=False) as destination:
                            temporary_path = Path(destination.name)
                            for block in iter(lambda: source.read(4 * 1024 * 1024), b""):
                                destination.write(block)
                    if checksum_file(temporary_path) != digest:
                        raise ValueError("Контрольная сумма загруженного файла не совпала.")
                    temporary_path.replace(target)
                    break
                except (OSError, ValueError) as exc:
                    error = exc
                finally:
                    if temporary_path is not None:
                        temporary_path.unlink(missing_ok=True)
            else:
                raise RuntimeError(f"Не удалось загрузить опубликованный файл {name}.") from error
        paths.append(target)
    return tuple(paths)


class JinaScores:
    def __init__(self, directory=None, threads=8):
        tokenizer_path, model_path = jina_checkpoint(directory)
        self.tokenizer = Tokenizer.from_file(str(tokenizer_path))
        self.start, self.end, self.pad = (self.tokenizer.token_to_id(token) for token in ("<s>", "</s>", "<pad>"))
        assert all(token is not None for token in (self.start, self.end, self.pad))
        options = ort.SessionOptions()
        options.intra_op_num_threads = max(1, threads)
        options.inter_op_num_threads = 1
        self.session = ort.InferenceSession(str(model_path), sess_options=options,
                                            providers=["CPUExecutionProvider"])
        assert {item.name for item in self.session.get_inputs()} == {"input_ids", "attention_mask"}

    def encode(self, pairs, limit=384):
        queries, documents = pairs
        encoded = []
        for first in range(0, len(queries), 512):
            q_ids = self.tokenizer.encode_batch(queries[first:first+512], add_special_tokens=False)
            d_ids = self.tokenizer.encode_batch(documents[first:first+512], add_special_tokens=False)
            for query, doc in zip(q_ids, d_ids):
                question = clip_tokens(query.ids, min(limit // 3, 128))
                passage = clip_tokens(doc.ids, limit - 4 - len(question))
                encoded.append([self.start, *question, self.end, self.end, *passage, self.end])
        return encoded

    def score_batches(self, pairs, limit=384, batch_size=16):
        items = self.encode(pairs, limit)
        lengths = np.fromiter((len(ids) for ids in items), dtype=np.int32, count=len(items))
        order = np.argsort(lengths, kind="stable")
        for start in range(0, len(order), batch_size):
            indices = order[start:start+batch_size]
            width = ((int(lengths[indices].max()) + 7) // 8) * 8
            ids = np.full((len(indices), width), self.pad, dtype=np.int64)
            mask = np.zeros_like(ids)
            for row, ix in enumerate(indices):
                tokens = items[ix]
                ids[row, :len(tokens)] = tokens
                mask[row, :len(tokens)] = 1
            output = self.session.run(None, {"input_ids": ids, "attention_mask": mask})[0].reshape(-1)
            if len(output) != len(indices) or not np.isfinite(output).all():
                raise ValueError("Jina вернула неверное число скоров или неконечные значения.")
            yield indices, output

    def predict(self, pairs, limit=384, batch_size=16):
        scores = np.empty(len(pairs[0]), dtype=np.float32)
        for indices, output in self.score_batches(pairs, limit, batch_size):
            scores[indices] = output
        return scores

## Обучение и итоговый скор

Веса объединения учим на 392 отложенных запросах из `train.csv`. Для прогноза на test переобучаем CatBoost на всём train. Перед объединением нормируем обе оценки внутри запроса.

In [ ]:
import numpy as np


import pandas as pd


from scipy.optimize import minimize


from scipy.special import expit


def calibration_split(frame, query_count=392):
    """Оставляет целые компоненты графа «запрос — фрагмент» для настройки реранкера."""
    components = query_components(frame)
    by_query = pd.DataFrame({"query": frame.query_id, "component": components}).drop_duplicates("query")
    sizes = by_query.component.value_counts(sort=False)
    pool = sizes[sizes <= 80]
    selected = []
    remaining = query_count
    for component in np.random.default_rng(20261107).permutation(pool.index.to_numpy()):
        n_queries = int(pool[component])
        if n_queries <= remaining:
            selected.append(component)
            remaining -= n_queries
        if not remaining:
            break
    if remaining:
        raise ValueError("Не удалось выделить заданное число запросов целыми компонентами.")
    holdout = np.isin(components, selected)
    fit, calib = [frame.loc[mask].reset_index(drop=True) for mask in (~holdout, holdout)]
    assert not set(fit.query_id).intersection(calib.query_id)
    assert not set(fit.document_id).intersection(calib.document_id)
    return fit, calib


def fit_fusion(frame, design, ridge=.05):
    """Линейный реранкер обучается на парах разной релевантности."""
    data = RelevancePairs(frame)
    x = design[data.pos_index] - design[data.neg_index]
    if not len(x):
        raise ValueError("Для обучения реранкера нужны пары разной релевантности.")
    order = data.groups.ranking(.75 * design[:,0] + .25 * design[:,1])
    discount = np.empty(order.shape, dtype=np.float64)
    values = DISCOUNTS[np.arange(order.shape[1]).clip(max=10)]
    np.put_along_axis(discount, order, values[None, :], axis=1)
    swap = np.abs(discount[data.group, data.positive] - discount[data.group, data.negative])
    weight = (.035 + swap) * data.inv_ideal / np.sqrt(np.bincount(data.group)[data.group])
    weight /= weight.sum()
    penalty = np.r_[[ridge, ridge], np.full(design.shape[1] - 2, .05)]

    def objective(coef):
        gap = x @ coef
        loss = weight @ np.logaddexp(0., -gap) + .5 * penalty @ (coef ** 2)
        grad = -(x.T @ (weight * expit(-gap))) + penalty * coef
        return loss, grad

    result = minimize(objective, np.ones(x.shape[1]), method="L-BFGS-B", jac=True,
                      bounds=[(0., None)] * x.shape[1],
                      options={"maxiter":150, "ftol":1e-12, "gtol":1e-9})
    if not result.success or not np.isfinite(result.x).all() or result.x.sum() <= 0:
        raise RuntimeError(f"Не удалось обучить реранкер. {result.message}")
    return result.x / result.x.sum()


def fusion_features(frame, semantic, lexical):
    neural = normalize_scores(frame, np.asarray(semantic, dtype=np.float64))
    tree = normalize_scores(frame, np.asarray(lexical, dtype=np.float64))
    return np.column_stack([neural, tree])

In [ ]:
started = monotonic()
fit, calibration = calibration_split(train)
scorer = JinaScores(threads=min(8, os.cpu_count() or 1))

In [ ]:
_, train_titles = rerank_texts(fit)
calibration_text, _ = rerank_texts(calibration, train_titles)
neural_calibration = scorer.predict(calibration_text)
print(f"Jina оценила {len(neural_calibration):,} пар для обучения реранкера", flush=True)

_, train_titles = rerank_texts(train)
test_text, _ = rerank_texts(test, train_titles)
neural_test = scorer.predict(test_text)
print(f"Jina оценила {len(neural_test):,} тестовых пар", flush=True)
del scorer

In [ ]:
lexical_calibration = classical_scores(fit, calibration)
lexical_test = classical_scores(train, test)

In [ ]:
calibration_design = fusion_features(calibration, neural_calibration, lexical_calibration)
weights = fit_fusion(calibration, calibration_design)
prediction = fusion_features(test, neural_test, lexical_test) @ weights

assert len(prediction) == len(test) and np.isfinite(prediction).all()
print(f"Веса реранкера — Jina {weights[0]:.3f}, CatBoost {weights[1]:.3f}")
print(f"Обучение и прогноз завершены за {(monotonic() - started) / 60:.1f} мин")

## Файл отправки

Записываем `id,prediction` в порядке строк `sample_submission.csv`.

In [ ]:
score_by_id = pd.Series(prediction, index=test.id, name="prediction")
submission = sample[["id"]].copy()
submission["prediction"] = score_by_id.reindex(submission.id).to_numpy()

assert submission.id.equals(sample.id)
assert len(submission) == len(test) and np.isfinite(submission.prediction).all()
submission.to_csv("submission.csv", index=False, float_format="%.10f")
print(f"Сохранён submission.csv. Строк — {len(submission):,}")
print(submission.head(3).to_string(index=False))